In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Read CSVs into DataFrames (these variables persist in the kernel)
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_logs_df = pd.read_csv('error_logs.csv')

# Helper to convert timestamp column (seconds) to timezone-aware datetime in Asia/Shanghai
def to_local_dt(series):
    # handle empty series
    if series.size == 0:
        return pd.to_datetime(series, unit='s')
    return pd.to_datetime(series, unit='s', utc=True).dt.tz_convert(tz)

# Helper to build compact summary dict for a dataframe
def summarize_df(df, name, field_col=None, field_label=None, top_n=5, max_list=50, is_error_logs=False):
    # Ensure timestamp conversion
    ts_col = 'timestamp'
    if ts_col in df.columns and not df.empty:
        ts_dt = to_local_dt(df[ts_col])
        df = df.copy()
        # human-readable timestamp column for display
        df['timestamp_dt'] = ts_dt.dt.strftime('%Y-%m-%d %H:%M:%S%z')
        min_ts = ts_dt.min()
        max_ts = ts_dt.max()
        min_ts_s = None if pd.isna(min_ts) else min_ts.isoformat()
        max_ts_s = None if pd.isna(max_ts) else max_ts.isoformat()
    else:
        # empty or no timestamp
        df = df.copy()
        df['timestamp_dt'] = pd.Series([], dtype=object)
        min_ts_s = None
        max_ts_s = None

    # cmdb_id list
    cmdb_ids = []
    if 'cmdb_id' in df.columns:
        cmdb_ids = sorted(df['cmdb_id'].dropna().unique().tolist())[:max_list]

    # fields: kpi_name / trace_name / log_name or 'message' for error_logs
    if is_error_logs:
        fields = ['message']
    else:
        fields = []
        if field_col and field_col in df.columns:
            fields = sorted(df[field_col].dropna().unique().tolist())[:max_list]

    # top rows (keep compact set of columns)
    if not df.empty:
        display_cols = [c for c in ['timestamp', 'timestamp_dt', 'cmdb_id', field_col, 'value'] if c in df.columns]
        top_rows = df.loc[:, display_cols].head(top_n)
    else:
        top_rows = df.head(0)

    summary = {
        'file': name,
        'rows': int(df.shape[0]),
        'cmdb_ids': cmdb_ids,
        'fields': fields,
        'min_timestamp': min_ts_s,
        'max_timestamp': max_ts_s,
        'top_rows': top_rows
    }
    return summary

metric_summary = summarize_df(metric_df, 'metric.csv', field_col='kpi_name')
trace_summary = summarize_df(trace_df, 'trace.csv', field_col='trace_name')
log_summary = summarize_df(log_df, 'log.csv', field_col='log_name')
error_logs_summary = summarize_df(error_logs_df, 'error_logs.csv', is_error_logs=True)

# Return the four compact summaries (IPython will display these variables)
metric_summary, trace_summary, log_summary, error_logs_summary

```
Out[1]:
```
Here’s a concise summary of the execution results:

- metric.csv
  - Total rows: 29,727
  - cmdb_id values (sorted sample / up to displayed): IG01, IG02, MG01, MG02, Mysql01, Mysql02, Redis01, Redis02, ServiceTest1…ServiceTest11, Tomcat01–Tomcat04, apache01, apache02, dockerA2
  - Example KPI names shown (truncated list): many JVM, MySQL, container and OS KPIs (e.g., JVM-Memory_7778_JVM_Memory_HeapMemoryMax, JVM-Memory_7778_JVM_Memory_HeapMemoryUsed, OSLinux-CPU_CPU-0_SingleCpuUtil, Container-..._CpuPercent, Mysql-MySQL_3306_Connections, etc.)
  - Time range: 2021-03-25 22:00:00+08 to 2021-03-25 22:30:00+08
  - Top 5 rows (examples): IG01 JVM memory and thread KPIs at 2021-03-25 22:00:00+08

- trace.csv
  - Total rows: 6,780
  - cmdb_id values: IG01, IG02, MG01, MG02, Tomcat01–Tomcat04, dockerA1–B2 (subset shown)
  - trace fields (examples): trace.self.duration_mean, trace.self.duration_p95, trace.self.row_count, trace.to_Tomcat01.duration_mean, trace.from_MG01.duration_mean, etc.
  - Time range: 2021-03-25 22:00:00+08 to 2021-03-25 22:30:00+08
  - Top 5 rows (examples): IG01 trace.self.* and trace.to_Tomcat01.* at 2021-03-25 22:00:00+08

- log.csv
  - Total rows: 397
  - cmdb_id values: Tomcat01, Tomcat02, Tomcat03, Tomcat04, apache01, apache02
  - log fields: log.error_count, log.gc_pause_max, log.row_count
  - Time range: 2021-03-25 22:00:00+08 to 2021-03-25 22:30:00+08
  - Top 5 rows (examples): Tomcat01/02/03 log.error_count = 0 and log.row_count values at 2021-03-25 22:00:00+08

- error_logs.csv
  - Total rows: 0 (empty)
  - No cmdb_id values, no timestamps
  - Field: message (but no records present)

All results were returned as compact summary dictionaries (each includes row count, sorted cmdb_id list, field names sample, min/max timestamps, and a top-5-row preview).

The original code execution output of IPython Kernel is also provided below for reference:

({'file': 'metric.csv', 'rows': 29727, 'cmdb_ids': ['IG01', 'IG02', 'MG01', 'MG02', 'Mysql01', 'Mysql02', 'Redis01', 'Redis02', 'ServiceTest1', 'ServiceTest10', 'ServiceTest11', 'ServiceTest2', 'ServiceTest3', 'ServiceTest4', 'ServiceTest5', 'ServiceTest6', 'ServiceTest7', 'ServiceTest8', 'ServiceTest9', 'Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04', 'apache01', 'apache02', 'dockerA2'], 'fields': ['Container-DOCKER_CONTAINER_2c2336e2994f--bcou-role-st-uat-statefulset-1--bcou--UATWKR06_CpuPercent', 'Container-DOCKER_CONTAINER_2c2336e2994f--bcou-role-st-uat-statefulset-1--bcou--UATWKR06_MemLimit', 'Container-DOCKER_CONTAINER_2c2336e2994f--bcou-role-st-uat-statefulset-1--bcou--UATWKR06_MemPercent', 'Container-DOCKER_CONTAINER_2c2336e2994f--bcou-role-st-uat-statefulset-1--bcou--UATWKR06_MemUsage', 'Container-DOCKER_CONTAINER_2c2336e2994f--bcou-role-st-uat-statefulset-1--bcou--UATWKR06_NetworkRxBytes', 'Container-DOCKER_CONTAINER_2c2336e2994f--bcou-role-st-uat-statefulset-1--bcou--UATWKR06_NetworkTxBytes', 'JVM-Memory_7778_JVM_Memory_HeapMemoryMax', 'JVM-Memory_7778_JVM_Memory_HeapMemoryUsage', 'JVM-Memory_7778_JVM_Memory_HeapMemoryUsed', 'JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed', 'JVM-Memory_7779_JVM_Memory_HeapMemoryMax', 'JVM-Memory_7779_JVM_Memory_HeapMemoryUsage', 'JVM-Memory_7779_JVM_Memory_HeapMemoryUsed', 'JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed', 'JVM-Operating System_7778_JVM_JVM_CPULoad', 'JVM-Operating System_7779_JVM_JVM_CPULoad', 'JVM-Runtime_7778_JVM_JVM_Uptime', 'JVM-Runtime_7779_JVM_JVM_Uptime', 'JVM-Threads_7778_JVM_ThreadCount_Threads', 'JVM-Threads_7779_JVM_ThreadCount_Threads', 'Mysql-MySQL_3306_Aborted Clients', 'Mysql-MySQL_3306_Aborted Connects', 'Mysql-MySQL_3306_Binlog Cache Disk Use', 'Mysql-MySQL_3306_Binlog Cache Use', 'Mysql-MySQL_3306_Binlog stmt cache disk use', 'Mysql-MySQL_3306_Binlog stmt cache use', 'Mysql-MySQL_3306_Bytes Received', 'Mysql-MySQL_3306_Bytes Sent', 'Mysql-MySQL_3306_Com Delete', 'Mysql-MySQL_3306_Com Delete Multi', 'Mysql-MySQL_3306_Com Insert', 'Mysql-MySQL_3306_Com Insert Select', 'Mysql-MySQL_3306_Com Load', 'Mysql-MySQL_3306_Com Replace', 'Mysql-MySQL_3306_Com Replace Select', 'Mysql-MySQL_3306_Com Select', 'Mysql-MySQL_3306_Com Update', 'Mysql-MySQL_3306_Com Update Multi', 'Mysql-MySQL_3306_Connections', 'Mysql-MySQL_3306_Created Tmp Disk Tables', 'Mysql-MySQL_3306_Created Tmp Files', 'Mysql-MySQL_3306_Created Tmp Tables', 'Mysql-MySQL_3306_CurrentSQLMaxRunningTime', 'Mysql-MySQL_3306_GetConnectedStateOfMysqld', 'Mysql-MySQL_3306_GetResponseTimeOfMysqld', 'Mysql-MySQL_3306_Handler Commit', 'Mysql-MySQL_3306_Handler Delete', 'Mysql-MySQL_3306_Handler Read First', 'Mysql-MySQL_3306_Handler Read Key', 'Mysql-MySQL_3306_Handler Read Next'], 'min_timestamp': '2021-03-25T22:00:00+08:00', 'max_timestamp': '2021-03-25T22:30:00+08:00', 'top_rows':     timestamp              timestamp_dt cmdb_id                                     kpi_name         value
0  1616680800  2021-03-25 22:00:00+0800    IG01     JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09
1  1616680800  2021-03-25 22:00:00+0800    IG01    JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  4.545795e+08
2  1616680800  2021-03-25 22:00:00+0800    IG01  JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.846119e+07
3  1616680800  2021-03-25 22:00:00+0800    IG01     JVM-Threads_7778_JVM_ThreadCount_Threads  6.800000e+01
4  1616680800  2021-03-25 22:00:00+0800    IG01              OSLinux-CPU_CPU-0_SingleCpuUtil  9.975000e-01}, {'file': 'trace.csv', 'rows': 6780, 'cmdb_ids': ['IG01', 'IG02', 'MG01', 'MG02', 'Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04', 'dockerA1', 'dockerA2', 'dockerB1', 'dockerB2'], 'fields': ['trace.from_IG01.duration_mean', 'trace.from_IG01.duration_p95', 'trace.from_IG01.row_count', 'trace.from_IG02.duration_mean', 'trace.from_IG02.duration_p95', 'trace.from_IG02.row_count', 'trace.from_MG01.duration_mean', 'trace.from_MG01.duration_p95', 'trace.from_MG01.row_count', 'trace.from_MG02.duration_mean', 'trace.from_MG02.duration_p95', 'trace.from_MG02.row_count', 'trace.from_Tomcat01.duration_mean', 'trace.from_Tomcat01.duration_p95', 'trace.from_Tomcat01.row_count', 'trace.from_Tomcat02.duration_mean', 'trace.from_Tomcat02.duration_p95', 'trace.from_Tomcat02.row_count', 'trace.from_Tomcat03.duration_mean', 'trace.from_Tomcat03.duration_p95', 'trace.from_Tomcat03.row_count', 'trace.from_Tomcat04.duration_mean', 'trace.from_Tomcat04.duration_p95', 'trace.from_Tomcat04.row_count', 'trace.from_dockerA1.duration_mean', 'trace.from_dockerA1.duration_p95', 'trace.from_dockerA1.row_count', 'trace.from_dockerA2.duration_mean', 'trace.from_dockerA2.duration_p95', 'trace.from_dockerA2.row_count', 'trace.from_dockerB1.duration_mean', 'trace.from_dockerB1.duration_p95', 'trace.from_dockerB1.row_count', 'trace.from_dockerB2.duration_mean', 'trace.from_dockerB2.duration_p95', 'trace.from_dockerB2.row_count', 'trace.self.duration_mean', 'trace.self.duration_p95', 'trace.self.row_count', 'trace.to_MG01.duration_mean', 'trace.to_MG01.duration_p95', 'trace.to_MG01.row_count', 'trace.to_MG02.duration_mean', 'trace.to_MG02.duration_p95', 'trace.to_MG02.row_count', 'trace.to_Tomcat01.duration_mean', 'trace.to_Tomcat01.duration_p95', 'trace.to_Tomcat01.row_count', 'trace.to_Tomcat02.duration_mean', 'trace.to_Tomcat02.duration_p95'], 'min_timestamp': '2021-03-25T22:00:00+08:00', 'max_timestamp': '2021-03-25T22:30:00+08:00', 'top_rows':     timestamp              timestamp_dt cmdb_id                       trace_name       value
0  1616680800  2021-03-25 22:00:00+0800    IG01         trace.self.duration_mean    0.164188
1  1616680800  2021-03-25 22:00:00+0800    IG01          trace.self.duration_p95    1.066000
2  1616680800  2021-03-25 22:00:00+0800    IG01             trace.self.row_count  528.000000
3  1616680800  2021-03-25 22:00:00+0800    IG01  trace.to_Tomcat01.duration_mean    0.159443
4  1616680800  2021-03-25 22:00:00+0800    IG01   trace.to_Tomcat01.duration_p95    1.041600}, {'file': 'log.csv', 'rows': 397, 'cmdb_ids': ['Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04', 'apache01', 'apache02'], 'fields': ['log.error_count', 'log.gc_pause_max', 'log.row_count'], 'min_timestamp': '2021-03-25T22:00:00+08:00', 'max_timestamp': '2021-03-25T22:30:00+08:00', 'top_rows':     timestamp              timestamp_dt   cmdb_id         log_name  value
0  1616680800  2021-03-25 22:00:00+0800  Tomcat01  log.error_count    0.0
1  1616680800  2021-03-25 22:00:00+0800  Tomcat01    log.row_count  161.0
2  1616680800  2021-03-25 22:00:00+0800  Tomcat02  log.error_count    0.0
3  1616680800  2021-03-25 22:00:00+0800  Tomcat02    log.row_count  138.0
4  1616680800  2021-03-25 22:00:00+0800  Tomcat03  log.error_count    0.0}, {'file': 'error_logs.csv', 'rows': 0, 'cmdb_ids': [], 'fields': ['message'], 'min_timestamp': None, 'max_timestamp': None, 'top_rows': Empty DataFrame
Columns: [timestamp, cmdb_id, message, timestamp_dt]
Index: []})```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz
tz = pytz.timezone('Asia/Shanghai')

# reuse metric_df from kernel
# 1) Compute global P95 and P05 and full_count per (cmdb_id, kpi_name)
thresholds_df = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], sort=True)['value']
    .agg(
        full_count = 'count',
        P95 = lambda s: s.quantile(0.95),
        P05 = lambda s: s.quantile(0.05)
    )
    .reset_index()
)

# convert thresholds index convenience (keep as DataFrame)
thresholds_df.shape, thresholds_df.head(3)

# 2) Filter metric.csv rows to the incident time window
start_ts = 1616680800
end_ts   = 1616682600
window_df = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()
window_df.shape

# 3) For each (cmdb_id, kpi_name) compute within-window aggregates
window_stats = (
    window_df
    .groupby(['cmdb_id', 'kpi_name'], sort=True)['value']
    .agg(
        window_count = 'count',
        window_max = 'max',
        window_min = 'min'
    )
    .reset_index()
)

# 4) Anomaly detection within the window: merge thresholds into window rows and flag anomalies
window_merged = window_df.merge(thresholds_df, on=['cmdb_id','kpi_name'], how='left', suffixes=('','_th'))
# create flags
window_merged['high_flag'] = window_merged['value'] >= window_merged['P95']
window_merged['low_flag']  = window_merged['value'] <= window_merged['P05']

# high anomaly per series
high_df = window_merged[window_merged['high_flag']]
high_stats = (
    high_df
    .groupby(['cmdb_id','kpi_name'], sort=True)
    .agg(
        high_anomaly_count = ('value','count'),
        high_first_ts = ('timestamp','min'),
        high_max = ('value','max')
    )
    .reset_index()
)

# low anomaly per series
low_df = window_merged[window_merged['low_flag']]
low_stats = (
    low_df
    .groupby(['cmdb_id','kpi_name'], sort=True)
    .agg(
        low_anomaly_count = ('value','count'),
        low_first_ts = ('timestamp','min'),
        low_min = ('value','min')
    )
    .reset_index()
)

# 5) Merge everything: thresholds_df (global), window_stats, high_stats, low_stats
summary = thresholds_df.merge(window_stats, on=['cmdb_id','kpi_name'], how='left')
summary = summary.merge(high_stats, on=['cmdb_id','kpi_name'], how='left')
summary = summary.merge(low_stats, on=['cmdb_id','kpi_name'], how='left')

# fill NaNs for counts with 0, keep numeric NaNs for min/max
summary['window_count'] = summary['window_count'].fillna(0).astype(int)
summary['high_anomaly_count'] = summary['high_anomaly_count'].fillna(0).astype(int)
summary['low_anomaly_count']  = summary['low_anomaly_count'].fillna(0).astype(int)

# 5) severity metrics
# ensure numeric columns exist
summary['high_max'] = summary['high_max'].astype(float)
summary['low_min']  = summary['low_min'].astype(float)
summary['P95'] = summary['P95'].astype(float)
summary['P05'] = summary['P05'].astype(float)

# compute severity percentages
summary['high_severity_pct'] = 0.0
mask_high = summary['high_anomaly_count'] > 0
summary.loc[mask_high, 'high_severity_pct'] = (
    (summary.loc[mask_high, 'high_max'] - summary.loc[mask_high, 'P95'])
    / summary.loc[mask_high, 'P95'].abs().replace({0:1e-9})
)

summary['low_severity_pct'] = 0.0
mask_low = summary['low_anomaly_count'] > 0
summary.loc[mask_low, 'low_severity_pct'] = (
    (summary.loc[mask_low, 'P05'] - summary.loc[mask_low, 'low_min'])
    / summary.loc[mask_low, 'P05'].abs().replace({0:1e-9})
)

# convert earliest anomaly timestamps to ISO strings in Asia/Shanghai, keep None when missing
def ts_to_iso(s):
    if pd.isna(s):
        return None
    return pd.to_datetime(int(s), unit='s', utc=True).tz_convert(tz).isoformat()

summary['high_first_ts'] = summary['high_first_ts'].apply(lambda x: ts_to_iso(x))
summary['low_first_ts']  = summary['low_first_ts'].apply(lambda x: ts_to_iso(x))

# 6) Prepare compact top-20 table sorted by max(high_anomaly_count, low_anomaly_count) desc
summary['max_anom_count'] = summary[['high_anomaly_count','low_anomaly_count']].max(axis=1)
# select and order columns requested
out_cols = [
    'cmdb_id','kpi_name','full_count','P95','P05',
    'window_count',
    'high_anomaly_count','high_first_ts','high_max','high_severity_pct',
    'low_anomaly_count','low_first_ts','low_min','low_severity_pct'
]
# ensure columns exist (some may be absent as NaN) - they do per construction
top20_summary = (
    summary.sort_values(by='max_anom_count', ascending=False)
    .loc[:, out_cols]
    .head(20)
    .reset_index(drop=True)
)

# keep full per-series threshold table and flagged window rows in kernel variables for later use
thresholds_per_series = thresholds_df.copy()        # full thresholds (P95/P05/full_count)
window_flagged_rows = window_merged.copy()          # window rows with P95/P05 and flags

# 7) total number of distinct (cmdb_id, kpi_name) series scanned
total_series = int(thresholds_df.shape[0])

# Display compact outputs: top20_summary (table) and total_series (single number)
top20_summary, total_series

```
Out[2]:
```
Summary (plain English):

- Total distinct (cmdb_id, kpi_name) series scanned: 1760.

- Top-20 series by anomaly counts (compact pattern observed):
  - Most of the top 20 are Mysql01 metrics (with a couple Tomcat02 / Redis02 entries).
  - For each top entry the global full_count = 31 and the window_count = 31 (the incident window had 31 samples for these series).
  - For these top rows both high_anomaly_count and low_anomaly_count = 31, and the earliest anomaly timestamp for both high and low is 2021-03-25T22:00:00+08:00.
  - In many cases P95 == P05 == the observed values in-window (so high_max == P95 and low_min == P05), which yields high_severity_pct and low_severity_pct ≈ 0.0.
  - One notable exception: Mysql01 -> "Mysql-MySQL_3306_Innodb data pending fsyncs" has P95 = 0 and high_max = 1, producing a very large high_severity_pct (artifact of dividing by a near-zero P95). This indicates P95 was zero historically so any positive value looks extreme numerically.

- Key interpretation and next steps:
  - Many series show identical P95/P05 and all window samples flagged as anomalous — likely these KPIs are constant outside/inside the window (or quantile equals observed value). Confirm whether quantile calculation on small sample sizes produced degenerate thresholds.
  - Investigate series with non-zero or extremely large severity_pct (e.g., the "data pending fsyncs" metric) because they may indicate meaningful deviation or a threshold artifact due to P95=0.
  - Use the saved kernel variables for deeper inspection:
    - thresholds_per_series (global P95/P05/full_count) and
    - window_flagged_rows (window rows with high/low flags and timestamp) are available for follow-up queries.

If you want, I can (from the kernel variables) list only the series that have non-zero severity_pct or P95 == 0 with non-zero window values for focused investigation.

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  full_count       P95       P05  window_count  high_anomaly_count              high_first_ts  high_max  high_severity_pct  low_anomaly_count               low_first_ts   low_min  low_severity_pct
0    Mysql01     Mysql-MySQL_3306_Innodb buffer pool read ahead          31       0.0       0.0            31                  31  2021-03-25T22:00:00+08:00       0.0       0.000000e+00                 31  2021-03-25T22:00:00+08:00       0.0               0.0
1    Mysql01  Mysql-MySQL_3306_Innodb buffer pool read ahead...          31       0.0       0.0            31                  31  2021-03-25T22:00:00+08:00       0.0       0.000000e+00                 31  2021-03-25T22:00:00+08:00       0.0               0.0
2    Mysql01  Mysql-MySQL_3306_Innodb buffer pool read ahead...          31       0.0       0.0            31                  31  2021-03-25T22:00:00+08:00       0.0       0.000000e+00                 31  2021-03-25T22:00:00+08:00       0.0               0.0
3   Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...          31     500.0     500.0            31                  31  2021-03-25T22:00:00+08:00     500.0       0.000000e+00                 31  2021-03-25T22:00:00+08:00     500.0               0.0
4    Mysql01        Mysql-MySQL_3306_Innodb data pending fsyncs          31       0.0       0.0            31                  31  2021-03-25T22:00:00+08:00       1.0       1.000000e+09                 30  2021-03-25T22:00:00+08:00       0.0               0.0
5    Mysql01      Mysql-MySQL_3306_Innodb buffer pool wait free          31       0.0       0.0            31                  31  2021-03-25T22:00:00+08:00       0.0       0.000000e+00                 31  2021-03-25T22:00:00+08:00       0.0               0.0
6    Mysql01     Mysql-MySQL_3306_Innodb buffer pool pages free          31  252175.0  252175.0            31                  31  2021-03-25T22:00:00+08:00  252175.0       0.000000e+00                 31  2021-03-25T22:00:00+08:00  252175.0               0.0
7    Mysql01         Mysql-MySQL_3306_Innodb data pending reads          31       0.0       0.0            31                  31  2021-03-25T22:00:00+08:00       0.0       0.000000e+00                 31  2021-03-25T22:00:00+08:00       0.0               0.0
8    Mysql01        Mysql-MySQL_3306_Innodb data pending writes          31       0.0       0.0            31                  31  2021-03-25T22:00:00+08:00       0.0       0.000000e+00                 31  2021-03-25T22:00:00+08:00       0.0               0.0
9    Mysql01                 Mysql-MySQL_3306_Handler Savepoint          31       0.0       0.0            31                  31  2021-03-25T22:00:00+08:00       0.0       0.000000e+00                 31  2021-03-25T22:00:00+08:00       0.0               0.0
10   Mysql01        Mysql-MySQL_3306_Handler Savepoint Rollback          31       0.0       0.0            31                  31  2021-03-25T22:00:00+08:00       0.0       0.000000e+00                 31  2021-03-25T22:00:00+08:00       0.0               0.0
11  Tomcat02     Tomcat-Sessions_7441--_SESSIONRejectedSessions          31       0.0       0.0            31                  31  2021-03-25T22:00:00+08:00       0.0       0.000000e+00                 31  2021-03-25T22:00:00+08:00       0.0               0.0
12  Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...          31       0.0       0.0            31                  31  2021-03-25T22:00:00+08:00       0.0       0.000000e+00                 31  2021-03-25T22:00:00+08:00       0.0               0.0
13  Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...          31       0.0       0.0            31                  31  2021-03-25T22:00:00+08:00       0.0       0.000000e+00                 31  2021-03-25T22:00:00+08:00       0.0               0.0
14   Mysql01             Mysql-MySQL_3306_Binlog stmt cache use          31     360.0     360.0            31                  31  2021-03-25T22:00:00+08:00     360.0       0.000000e+00                 31  2021-03-25T22:00:00+08:00     360.0               0.0
15   Mysql01                  Mysql-MySQL_3306_Com Delete Multi          31       0.0       0.0            31                  31  2021-03-25T22:00:00+08:00       0.0       0.000000e+00                 31  2021-03-25T22:00:00+08:00       0.0               0.0
16   Mysql01          Mysql-MySQL_3306_CurrentSQLMaxRunningTime          31       0.0       0.0            31                  31  2021-03-25T22:00:00+08:00       0.0       0.000000e+00                 31  2021-03-25T22:00:00+08:00       0.0               0.0
17   Mysql01         Mysql-MySQL_3306_GetConnectedStateOfMysqld          31       1.0       1.0            31                  31  2021-03-25T22:00:00+08:00       1.0       0.000000e+00                 31  2021-03-25T22:00:00+08:00       1.0               0.0
18   Mysql01           Mysql-MySQL_3306_GetResponseTimeOfMysqld          31       0.0       0.0            31                  31  2021-03-25T22:00:00+08:00       0.0       0.000000e+00                 31  2021-03-25T22:00:00+08:00       0.0               0.0
19   Redis02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount          31       4.0       4.0            31                  31  2021-03-25T22:00:00+08:00       4.0       0.000000e+00                 31  2021-03-25T22:00:00+08:00       4.0               0.0, 1760)```
```

In [ ]:
# In[3]:

# Reuse existing kernel variables: summary, thresholds_per_series, window_flagged_rows
import pandas as pd
import pytz

# Work on a copy to preserve original 'summary'
candidate_df = summary.copy()

# Ensure numeric fields have no NaN for comparisons
candidate_df['high_severity_pct'] = candidate_df['high_severity_pct'].fillna(0.0).astype(float)
candidate_df['low_severity_pct']  = candidate_df['low_severity_pct'].fillna(0.0).astype(float)
candidate_df['P95'] = candidate_df['P95'].fillna(0.0).astype(float)
candidate_df['window_max'] = candidate_df.get('window_max', pd.Series(0, index=candidate_df.index)).fillna(0.0).astype(float)

# Filter condition: meaningful anomalies
mask = (
    (candidate_df['high_severity_pct'] > 0.0) |
    (candidate_df['low_severity_pct']  > 0.0) |
    ((candidate_df['P95'] == 0.0) & (candidate_df['window_max'] > 0.0))
)

candidates_full = candidate_df[mask].copy()   # store full result for later inspection

# Ranking key and top-20 selection
candidates_full['max_severity'] = candidates_full[['high_severity_pct','low_severity_pct']].max(axis=1)
candidates_top20 = (
    candidates_full
    .sort_values(by='max_severity', ascending=False)
    .loc[:, [
        'cmdb_id','kpi_name','full_count','P95','P05','window_count',
        'high_anomaly_count','high_first_ts','high_max','high_severity_pct',
        'low_anomaly_count','low_first_ts','low_min','low_severity_pct'
    ]]
    .head(20)
    .reset_index(drop=True)
)

total_matches = int(candidates_full.shape[0])

# Display compact results (variables)
candidates_top20, total_matches

```
Out[3]:
```
Summary (plain English):

- Total candidate series matching the filter: 711.

- Top findings (from the top-20 results):
  - Many top candidates are Mysql metrics where P95 == 0 but window_max > 0 (e.g., Mysql01 -> "max trx lock memory bytes" and several Innodb metrics). These produce extremely large computed high_severity_pct because the historical P95 is zero — likely a threshold artifact (any positive value looks huge relative to zero).
  - Several non-MySQL series show meaningful, non-artifactual severity percentages:
    - ServiceTest9 app.mrt — high_max ~2.3647, high_severity_pct ≈ 12.46, low_severity_pct ≈ 0.149. Two high points flagged; earliest high at 2021-03-25T22:02:00+08.
    - Tomcat01 OSLinux-CPU_CPU_CPUWio — high_max ~1.8833, high_severity_pct ≈ 8.85, low_severity_pct ~0.0012. Two high points flagged; earliest high at 2021-03-25T22:01:00+08.
    - ServiceTest8 and ServiceTest5 app.mrt — also show elevated high_severity_pct (~6.57 and ~6.55 respectively) with 2 high points each.
    - Tomcat01 disk metrics (DSKRead, DSKRTps) and some Redis/ Tomcat network metrics show single high points with moderate severity (high_severity_pct in the ~2–3 range).
  - Several network/process counters (TCP-FIN-WAIT, CLOSE-WAIT, etc.) appear because historical P95 is zero and a small number of nonzero samples occurred in the window — again these may be artifacts or meaningful depending on context.

- Interpretation / recommended next steps:
  1. Prioritize series with non-zero P95 and substantial high_severity_pct (e.g., ServiceTest9, ServiceTest8, ServiceTest5, Tomcat01 CPU/IO) — these look like real deviations.
  2. For series where P95 == 0 but window_max > 0 (many MySQL and some network/process counters), manually inspect raw time series and thresholds: small baseline values can make the computed severity metric misleading. Decide whether to treat these as real alerts or threshold-artifacts.
  3. Use the stored variables (thresholds_per_series and window_flagged_rows) to pull raw flagged samples and verify timestamps/values, and cross-check logs/traces for correlated errors around the earliest flagged times (timestamps shown in the table).

If you want, I can now:
- show only the non-artifact candidates (P95 > 0) with non-zero severity, or
- display the flagged raw samples for a selected series (e.g., ServiceTest9 or a Mysql01 metric) for deeper inspection.

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id                                           kpi_name  full_count         P95       P05  window_count  high_anomaly_count              high_first_ts    high_max  high_severity_pct  low_anomaly_count               low_first_ts   low_min  low_severity_pct
0        Mysql01         Mysql-MySQL_3306_max trx lock memory bytes          31    0.000000  0.000000            31                  31  2021-03-25T22:00:00+08:00  1136.00000       1.136000e+12                 30  2021-03-25T22:00:00+08:00  0.000000          0.000000
1       Tomcat03       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT          30    0.000000  0.000000            30                  30  2021-03-25T22:00:00+08:00     1.00000       1.000000e+09                 29  2021-03-25T22:00:00+08:00  0.000000          0.000000
2        Mysql02              Mysql-MySQL_3306_LongestTrxActiveTime          31    0.000000  0.000000            31                  31  2021-03-25T22:00:00+08:00     1.00000       1.000000e+09                 30  2021-03-25T22:00:00+08:00  0.000000          0.000000
3        Mysql02          Mysql-MySQL_3306_CurrentSQLMaxRunningTime          31    0.000000  0.000000            31                  31  2021-03-25T22:00:00+08:00     1.00000       1.000000e+09                 30  2021-03-25T22:00:00+08:00  0.000000          0.000000
4        Mysql01               Mysql-MySQL_3306_Max trx rows locked          31    0.000000  0.000000            31                  31  2021-03-25T22:00:00+08:00     1.00000       1.000000e+09                 30  2021-03-25T22:00:00+08:00  0.000000          0.000000
5        Mysql01                Mysql-MySQL_3306_MaxTrxRowsModified          31    0.000000  0.000000            31                  31  2021-03-25T22:00:00+08:00     1.00000       1.000000e+09                 30  2021-03-25T22:00:00+08:00  0.000000          0.000000
6        Mysql01      Mysql-MySQL_3306_Innodb os log pending fsyncs          31    0.000000  0.000000            31                  31  2021-03-25T22:00:00+08:00     1.00000       1.000000e+09                 30  2021-03-25T22:00:00+08:00  0.000000          0.000000
7        Mysql01        Mysql-MySQL_3306_Innodb data pending fsyncs          31    0.000000  0.000000            31                  31  2021-03-25T22:00:00+08:00     1.00000       1.000000e+09                 30  2021-03-25T22:00:00+08:00  0.000000          0.000000
8           MG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT          30    0.000000  0.000000            30                  30  2021-03-25T22:00:00+08:00     1.00000       1.000000e+09                 29  2021-03-25T22:00:00+08:00  0.000000          0.000000
9   ServiceTest9                                            app.mrt          31    0.175665  0.021160            31                   2  2021-03-25T22:02:00+08:00     2.36474       1.246165e+01                  2  2021-03-25T22:21:00+08:00  0.018000          0.149338
10      Tomcat01                             OSLinux-CPU_CPU_CPUWio          31    0.191228  0.012486            31                   2  2021-03-25T22:01:00+08:00     1.88330       8.848427e+00                  2  2021-03-25T22:07:00+08:00  0.012471          0.001201
11  ServiceTest8                                            app.mrt          31    0.137610  0.034815            31                   2  2021-03-25T22:01:00+08:00     1.04200       6.572124e+00                  2  2021-03-25T22:00:00+08:00  0.025000          0.281919
12  ServiceTest5                                            app.mrt          31    0.095915  0.018260            31                   2  2021-03-25T22:05:00+08:00     0.72388       6.547099e+00                  2  2021-03-25T22:11:00+08:00  0.017450          0.044359
13       Redis01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT          18    0.150000  0.000000            18                   1  2021-03-25T22:13:00+08:00     1.00000       5.666667e+00                 17  2021-03-25T22:02:00+08:00  0.000000          0.000000
14       Redis02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...          16    0.100000  0.000000            16                   1  2021-03-25T22:30:00+08:00     0.40000       3.000000e+00                 15  2021-03-25T22:00:00+08:00  0.000000          0.000000
15       Redis02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps          16    0.250000  0.000000            16                   1  2021-03-25T22:02:00+08:00     1.00000       3.000000e+00                 15  2021-03-25T22:00:00+08:00  0.000000          0.000000
16      Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead          16  106.233325  0.000000            16                   1  2021-03-25T22:22:00+08:00   406.93330       2.830562e+00                 13  2021-03-25T22:00:00+08:00  0.000000          0.000000
17      Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps          16    1.687500  0.000000            16                   1  2021-03-25T22:22:00+08:00     6.20000       2.674075e+00                 13  2021-03-25T22:00:00+08:00  0.000000          0.000000
18      Tomcat02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT          30    0.550000  0.000000            30                   2  2021-03-25T22:04:00+08:00     2.00000       2.636364e+00                 28  2021-03-25T22:00:00+08:00  0.000000          0.000000
19      Tomcat01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT          30    0.550000  0.000000            30                   2  2021-03-25T22:01:00+08:00     2.00000       2.636364e+00                 28  2021-03-25T22:00:00+08:00  0.000000          0.000000, 711)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing kernel variables: summary (per-series merged thresholds+window stats+severity) and window_merged (window rows with flags)
# Filter series: P95 > 0 and max(high_severity_pct, low_severity_pct) > 0
cand = summary.copy()
cand['max_sev'] = cand[['high_severity_pct','low_severity_pct']].max(axis=1)
cand_filtered = cand[(cand['P95'] > 0) & (cand['max_sev'] > 0)].reset_index(drop=True)

# Helper to compute streak stats for a series given window_merged rows
def compute_streaks(df_series, flag_col):
    # df_series expected sorted by timestamp ascending and contain 'timestamp' and flag_col boolean
    ts_arr = df_series.loc[df_series[flag_col], 'timestamp'].values.astype('int64')
    if ts_arr.size == 0:
        return {
            'anomaly_count': 0,
            'first_ts': None,
            'last_ts': None,
            'max_val': np.nan,
            'streak_max': 0,
            'streak_first_ts': None
        }
    # anomaly_count, first/last timestamps, max value among flagged rows
    anomaly_count = ts_arr.size
    first_ts = pd.to_datetime(int(ts_arr.min()), unit='s', utc=True).tz_convert(tz).isoformat()
    last_ts  = pd.to_datetime(int(ts_arr.max()), unit='s', utc=True).tz_convert(tz).isoformat()
    max_val = float(df_series.loc[df_series[flag_col], 'value'].max())
    # compute consecutive runs where successive flagged timestamps differ by exactly 60 seconds
    if ts_arr.size == 1:
        return {
            'anomaly_count': anomaly_count,
            'first_ts': first_ts,
            'last_ts': last_ts,
            'max_val': max_val,
            'streak_max': 1,
            'streak_first_ts': first_ts
        }
    diffs = np.diff(ts_arr)
    is_consec = diffs == 60
    # build group ids: start at 0, increment when not consecutive
    group_ids = np.empty(ts_arr.size, dtype=int)
    group_ids[0] = 0
    # (~is_consec).cumsum() gives cumulative count of breaks; assign to positions 1:
    group_ids[1:] = (~is_consec).cumsum()
    # compute sizes per group
    counts = np.bincount(group_ids)
    streak_max = int(counts.max())
    # find group index of longest streak (choose earliest if multiple)
    max_group_idx = int(np.nonzero(counts == counts.max())[0][0])
    # earliest timestamp of that group
    group_ts = ts_arr[group_ids == max_group_idx]
    streak_first_ts = pd.to_datetime(int(group_ts.min()), unit='s', utc=True).tz_convert(tz).isoformat()
    return {
        'anomaly_count': int(anomaly_count),
        'first_ts': first_ts,
        'last_ts': last_ts,
        'max_val': max_val,
        'streak_max': streak_max,
        'streak_first_ts': streak_first_ts
    }

# Iterate over filtered series and compute requested fields
rows = []
# For performance, convert window_merged to grouped object by series
wm = window_merged.copy()
# ensure timestamp is int
wm['timestamp'] = wm['timestamp'].astype(int)
wm_groups = wm.groupby(['cmdb_id','kpi_name'], sort=False)

for _, r in cand_filtered.iterrows():
    cm = r['cmdb_id']
    kp = r['kpi_name']
    key = (cm, kp)
    # get all window rows for this series (may be empty)
    try:
        grp = wm_groups.get_group(key).sort_values('timestamp', ascending=True)
    except KeyError:
        grp = pd.DataFrame(columns=wm.columns)  # empty
    # high stats via compute_streaks on 'high_flag'
    high_stats = compute_streaks(grp, 'high_flag')
    low_stats  = compute_streaks(grp, 'low_flag')
    # assemble result row
    rows.append({
        'cmdb_id': cm,
        'kpi_name': kp,
        'full_count': int(r['full_count']),
        'P95': float(r['P95']),
        'P05': float(r['P05']),
        'window_count': int(r['window_count']),
        'high_anomaly_count': high_stats['anomaly_count'],
        'high_first_ts': high_stats['first_ts'],
        'high_last_ts': high_stats['last_ts'],
        'high_max': high_stats['max_val'],
        'high_streak_max': high_stats['streak_max'],
        'high_streak_first_ts': high_stats['streak_first_ts'],
        'low_anomaly_count': low_stats['anomaly_count'],
        'low_first_ts': low_stats['first_ts'],
        'low_last_ts': low_stats['last_ts'],
        'low_min': low_stats['max_val'] if np.isnan(low_stats['max_val']) else float(low_stats['max_val']),  # max_val holds min for low branch naming consistency below
        'low_streak_max': low_stats['streak_max'],
        'low_streak_first_ts': low_stats['streak_first_ts'],
        # include ranking key
        'max_severity': float(r['max_sev'])
    })

# Build DataFrame
fault_streaks_df = pd.DataFrame(rows)

# Note: low_stats['max_val'] currently holds max of values among low-flagged rows; we need low_min (min value among low flagged rows)
# Recompute low_min properly from window_merged for those series
if not fault_streaks_df.empty:
    # map low_min from window_merged grouped min where low_flag True
    low_min_map = (
        wm[wm['low_flag']]
        .groupby(['cmdb_id','kpi_name'], sort=False)['value']
        .min()
    )
    # create index matching keys
    def get_low_min(cm, kp):
        return float(low_min_map.get((cm, kp), np.nan))
    fault_streaks_df['low_min'] = fault_streaks_df.apply(lambda x: get_low_min(x['cmdb_id'], x['kpi_name']), axis=1)

# Sort by severity descending and select top 20
top20_streaks = fault_streaks_df.sort_values(by='max_severity', ascending=False).head(20).reset_index(drop=True)

# Short list of likely faults: where high_streak_max >=2 or low_streak_max >=2 (up to 10 entries)
likely_faults = fault_streaks_df[
    (fault_streaks_df['high_streak_max'] >= 2) | (fault_streaks_df['low_streak_max'] >= 2)
].sort_values(by=['high_streak_max','low_streak_max','max_severity'], ascending=[False,False,False])

likely_faults_short = likely_faults.loc[:, ['cmdb_id','kpi_name','high_streak_max','low_streak_max']].head(10).reset_index(drop=True)

# Outputs to display: top20_streaks (compact table) and likely_faults_short (short list)
top20_streaks, likely_faults_short

```
Out[4]:
```
Summary (plain English):

- Scope: we examined series with P95 > 0 and non-zero severity within the incident window; the top-20 results are shown in the kernel table (compact), with per-series streak statistics computed.

- Top candidates (high severity / non-zero P95) — highlights from the top-20:
  - ServiceTest9 / kpi app.mrt — strong high severity (max severity ≈ 12.46). 31 samples in-window, 2 high points (first 2021-03-25T22:02:00+08), high_max = 2.36474. Also has low anomalies later in the window.
  - Tomcat01 / OSLinux-CPU_CPU_CPUWio — max severity ≈ 8.85. 2 high points (first 22:01), high_max ≈ 1.8833.
  - ServiceTest8 / app.mrt and ServiceTest5 / app.mrt — both show elevated high_severity_pct (≈ 6.57 and 6.55) with 2 high points each.
  - Redis01 / OSLinux-OSLinux_NETWORK_TCP-FIN-WAIT and several Tomcat/Redis/local-disk metrics also appear with measurable severity and single-point or small streak anomalies.
  - Several disk read/RTps metrics on Tomcat nodes show single high points but moderate severity.

- Fault-like series with consecutive anomaly runs:
  - We extracted series having streaks >= 2 (likely faults). Up to 10 are listed (short list from kernel):
    1. ServiceTest1 — app.rr (high_streak_max 29, low_streak_max 31)
    2. ServiceTest9 — app.sr (high_streak_max 28, low_streak_max 31)
    3. ServiceTest9 — app.rr (high_streak_max 28, low_streak_max 31)
    4. ServiceTest6 — app.rr (high_streak_max 27)
    5. Mysql02 — Mysql-MySQL_3306_Open Tables (high_streak_max 26, low_streak_max 2)
    6. ServiceTest5 — app.rr (high_streak_max 25, low_streak_max 31)
    7. ServiceTest7 — app.rr (high_streak_max 25)
    8. ServiceTest4 — app.rr (high_streak_max 25)
    9. Mysql02 — Mysql-MySQL_3306_Sort Rows (high_streak_max 4, low_streak_max 2)
    10. Tomcat03 — Tomcat-Threads_..._CurrentThreads... (high_streak_max 3, low_streak_max 6)

- Interpretation / recommended next steps:
  1. Prioritize series with non-zero P95 and non-trivial streaks (high_streak_max >= 2 or low_streak_max >= 2) — the short list above is the immediate candidate set for faults.
  2. For the highest-severity, non-artifact series (e.g., ServiceTest9/app.mrt, Tomcat01 CPU WIO, ServiceTest8/5 app.mrt), pull the raw flagged samples (timestamps and values) from window_flagged_rows and correlate with logs/traces around the earliest flagged time.
  3. For series with very long streaks (ServiceTest* app.rr/app.sr), confirm whether these reflect real sustained degradation or metric/collection issues (e.g., constant elevated values due to topology or instrumentation).
  4. Use thresholds_per_series and window_flagged_rows (available in the kernel) to inspect the raw points and to validate whether P95 calculation or small-sample effects are causing misleading severity numbers.

If you want, I can now:
- display the raw flagged samples for any one of the listed series (e.g., ServiceTest9/app.mrt or Mysql02/Open Tables), or
- return the full fault-streak table stored in the kernel for further slicing.

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id                                           kpi_name  full_count         P95       P05  window_count  high_anomaly_count              high_first_ts               high_last_ts    high_max  high_streak_max       high_streak_first_ts  low_anomaly_count               low_first_ts                low_last_ts   low_min  low_streak_max        low_streak_first_ts  max_severity
0   ServiceTest9                                            app.mrt          31    0.175665  0.021160            31                   2  2021-03-25T22:02:00+08:00  2021-03-25T22:30:00+08:00    2.364740                1  2021-03-25T22:02:00+08:00                  2  2021-03-25T22:21:00+08:00  2021-03-25T22:24:00+08:00  0.018000               1  2021-03-25T22:21:00+08:00     12.461646
1       Tomcat01                             OSLinux-CPU_CPU_CPUWio          31    0.191228  0.012486            31                   2  2021-03-25T22:01:00+08:00  2021-03-25T22:22:00+08:00    1.883300                1  2021-03-25T22:01:00+08:00                  2  2021-03-25T22:07:00+08:00  2021-03-25T22:17:00+08:00  0.012471               1  2021-03-25T22:07:00+08:00      8.848427
2   ServiceTest8                                            app.mrt          31    0.137610  0.034815            31                   2  2021-03-25T22:01:00+08:00  2021-03-25T22:17:00+08:00    1.042000                1  2021-03-25T22:01:00+08:00                  2  2021-03-25T22:00:00+08:00  2021-03-25T22:02:00+08:00  0.025000               1  2021-03-25T22:00:00+08:00      6.572124
3   ServiceTest5                                            app.mrt          31    0.095915  0.018260            31                   2  2021-03-25T22:05:00+08:00  2021-03-25T22:27:00+08:00    0.723880                1  2021-03-25T22:05:00+08:00                  2  2021-03-25T22:11:00+08:00  2021-03-25T22:25:00+08:00  0.017450               1  2021-03-25T22:11:00+08:00      6.547099
4        Redis01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT          18    0.150000  0.000000            18                   1  2021-03-25T22:13:00+08:00  2021-03-25T22:13:00+08:00    1.000000                1  2021-03-25T22:13:00+08:00                 17  2021-03-25T22:02:00+08:00  2021-03-25T22:30:00+08:00  0.000000               2  2021-03-25T22:02:00+08:00      5.666667
5        Redis02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...          16    0.100000  0.000000            16                   1  2021-03-25T22:30:00+08:00  2021-03-25T22:30:00+08:00    0.400000                1  2021-03-25T22:30:00+08:00                 15  2021-03-25T22:00:00+08:00  2021-03-25T22:28:00+08:00  0.000000               1  2021-03-25T22:00:00+08:00      3.000000
6        Redis02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps          16    0.250000  0.000000            16                   1  2021-03-25T22:02:00+08:00  2021-03-25T22:02:00+08:00    1.000000                1  2021-03-25T22:02:00+08:00                 15  2021-03-25T22:00:00+08:00  2021-03-25T22:30:00+08:00  0.000000               1  2021-03-25T22:00:00+08:00      3.000000
7       Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead          16  106.233325  0.000000            16                   1  2021-03-25T22:22:00+08:00  2021-03-25T22:22:00+08:00  406.933300                1  2021-03-25T22:22:00+08:00                 13  2021-03-25T22:00:00+08:00  2021-03-25T22:28:00+08:00  0.000000               1  2021-03-25T22:00:00+08:00      2.830562
8       Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps          16    1.687500  0.000000            16                   1  2021-03-25T22:22:00+08:00  2021-03-25T22:22:00+08:00    6.200000                1  2021-03-25T22:22:00+08:00                 13  2021-03-25T22:00:00+08:00  2021-03-25T22:28:00+08:00  0.000000               1  2021-03-25T22:00:00+08:00      2.674075
9       Tomcat02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT          30    0.550000  0.000000            30                   2  2021-03-25T22:04:00+08:00  2021-03-25T22:05:00+08:00    2.000000                2  2021-03-25T22:04:00+08:00                 28  2021-03-25T22:00:00+08:00  2021-03-25T22:30:00+08:00  0.000000              22  2021-03-25T22:09:00+08:00      2.636364
10      Tomcat01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT          30    0.550000  0.000000            30                   2  2021-03-25T22:01:00+08:00  2021-03-25T22:04:00+08:00    2.000000                1  2021-03-25T22:01:00+08:00                 28  2021-03-25T22:00:00+08:00  2021-03-25T22:30:00+08:00  0.000000              24  2021-03-25T22:05:00+08:00      2.636364
11       Redis01                             OSLinux-CPU_CPU_CPUWio          18    0.056985  0.007685            18                   1  2021-03-25T22:23:00+08:00  2021-03-25T22:23:00+08:00    0.191200                1  2021-03-25T22:23:00+08:00                  1  2021-03-25T22:17:00+08:00  2021-03-25T22:17:00+08:00  0.004200               1  2021-03-25T22:17:00+08:00      2.355269
12       Redis02                             OSLinux-CPU_CPU_CPUWio          31    0.043715  0.004168            31                   2  2021-03-25T22:00:00+08:00  2021-03-25T22:19:00+08:00    0.141738                1  2021-03-25T22:00:00+08:00                  2  2021-03-25T22:03:00+08:00  2021-03-25T22:16:00+08:00  0.004166               1  2021-03-25T22:03:00+08:00      2.242282
13  ServiceTest1                                            app.mrt          31    0.152390  0.028500            31                   2  2021-03-25T22:01:00+08:00  2021-03-25T22:27:00+08:00    0.469030                1  2021-03-25T22:01:00+08:00                  2  2021-03-25T22:08:00+08:00  2021-03-25T22:17:00+08:00  0.012530               1  2021-03-25T22:08:00+08:00      2.077827
14      apache02                            OSLinux-CPU_CPU_CPULoad          19    0.013000  0.000000            19                   1  2021-03-25T22:28:00+08:00  2021-03-25T22:28:00+08:00    0.040000                1  2021-03-25T22:28:00+08:00                 17  2021-03-25T22:01:00+08:00  2021-03-25T22:30:00+08:00  0.000000               2  2021-03-25T22:01:00+08:00      2.076923
15          MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead          14  105.773345  0.000000            14                   1  2021-03-25T22:01:00+08:00  2021-03-25T22:01:00+08:00  291.066700                1  2021-03-25T22:01:00+08:00                 12  2021-03-25T22:03:00+08:00  2021-03-25T22:27:00+08:00  0.000000               1  2021-03-25T22:03:00+08:00      1.751796
16       Mysql01                         OSLinux-CPU_CPU_CPUSysTime          19    0.535680  0.260510            19                   1  2021-03-25T22:10:00+08:00  2021-03-25T22:10:00+08:00    1.242900                1  2021-03-25T22:10:00+08:00                  1  2021-03-25T22:20:00+08:00  2021-03-25T22:20:00+08:00  0.212000               1  2021-03-25T22:20:00+08:00      1.320228
17      Tomcat04                             OSLinux-CPU_CPU_CPUWio          31    0.299271  0.022850            31                   2  2021-03-25T22:10:00+08:00  2021-03-25T22:20:00+08:00    0.677642                1  2021-03-25T22:10:00+08:00                  2  2021-03-25T22:09:00+08:00  2021-03-25T22:22:00+08:00  0.016600               1  2021-03-25T22:09:00+08:00      1.264309
18      Tomcat02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead          15   13.080000  0.000000            15                   1  2021-03-25T22:15:00+08:00  2021-03-25T22:15:00+08:00   29.600000                1  2021-03-25T22:15:00+08:00                 10  2021-03-25T22:02:00+08:00  2021-03-25T22:29:00+08:00  0.000000               1  2021-03-25T22:02:00+08:00      1.262997
19      Tomcat03     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps          14    5.500000  0.000000            14                   1  2021-03-25T22:01:00+08:00  2021-03-25T22:01:00+08:00   12.000000                1  2021-03-25T22:01:00+08:00                  9  2021-03-25T22:03:00+08:00  2021-03-25T22:27:00+08:00  0.000000               1  2021-03-25T22:03:00+08:00      1.181818,         cmdb_id                                           kpi_name  high_streak_max  low_streak_max
0  ServiceTest1                                             app.rr               29              31
1  ServiceTest9                                             app.sr               28              31
2  ServiceTest9                                             app.rr               28              31
3  ServiceTest6                                             app.rr               27               1
4       Mysql02                       Mysql-MySQL_3306_Open Tables               26               2
5  ServiceTest5                                             app.rr               25              31
6  ServiceTest7                                             app.rr               25               1
7  ServiceTest4                                             app.rr               25               1
8       Mysql02                         Mysql-MySQL_3306_Sort Rows                4               2
9      Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...                3               6)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse trace_df from kernel

# 1) global P95, P05, full_count per (cmdb_id, trace_name)
trace_thresholds = (
    trace_df
    .groupby(['cmdb_id','trace_name'], sort=True)['value']
    .agg(
        full_count = 'count',
        P95 = lambda s: float(s.quantile(0.95)),
        P05 = lambda s: float(s.quantile(0.05))
    )
    .reset_index()
)

# 2) filter to incident window
start_ts = 1616680800
end_ts   = 1616682600
trace_window = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)].copy()

# 3) merge thresholds and flag anomalies
trace_window = trace_window.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')
trace_window['high_flag'] = trace_window['value'] >= trace_window['P95']
trace_window['low_flag']  = trace_window['value'] <= trace_window['P05']

# helper to iso-format timestamps in Asia/Shanghai
def ts_to_iso(v):
    if pd.isna(v):
        return None
    return pd.to_datetime(int(v), unit='s', utc=True).tz_convert(tz).isoformat()

# high and low aggregations within window
high_stats = (
    trace_window[trace_window['high_flag']]
    .groupby(['cmdb_id','trace_name'], sort=True)
    .agg(
        high_anomaly_count = ('value','count'),
        high_first_ts = ('timestamp','min'),
        high_max = ('value','max')
    )
    .reset_index()
)

low_stats = (
    trace_window[trace_window['low_flag']]
    .groupby(['cmdb_id','trace_name'], sort=True)
    .agg(
        low_anomaly_count = ('value','count'),
        low_first_ts = ('timestamp','min'),
        low_min = ('value','min')
    )
    .reset_index()
)

# window counts and min/max (for completeness)
win_minmax = (
    trace_window
    .groupby(['cmdb_id','trace_name'], sort=True)['value']
    .agg(window_count='count', window_max='max', window_min='min')
    .reset_index()
)

# 4) merge everything into per-series summary
trace_summary = trace_thresholds.merge(win_minmax, on=['cmdb_id','trace_name'], how='left')
trace_summary = trace_summary.merge(high_stats, on=['cmdb_id','trace_name'], how='left')
trace_summary = trace_summary.merge(low_stats, on=['cmdb_id','trace_name'], how='left')

# fill counts with 0 where missing
trace_summary['window_count'] = trace_summary['window_count'].fillna(0).astype(int)
trace_summary['high_anomaly_count'] = trace_summary['high_anomaly_count'].fillna(0).astype(int)
trace_summary['low_anomaly_count']  = trace_summary['low_anomaly_count'].fillna(0).astype(int)

# convert timestamps to ISO strings
trace_summary['high_first_ts'] = trace_summary['high_first_ts'].apply(lambda x: ts_to_iso(x) if not pd.isna(x) else None)
trace_summary['low_first_ts']  = trace_summary['low_first_ts'].apply(lambda x: ts_to_iso(x) if not pd.isna(x) else None)

# ensure numeric columns
trace_summary['high_max'] = trace_summary['high_max'].astype(float)
trace_summary['low_min']  = trace_summary['low_min'].astype(float)
trace_summary['P95'] = trace_summary['P95'].astype(float)
trace_summary['P05'] = trace_summary['P05'].astype(float)

# compute severity percentages
trace_summary['high_severity_pct'] = 0.0
mask_h = trace_summary['high_anomaly_count'] > 0
trace_summary.loc[mask_h, 'high_severity_pct'] = (
    (trace_summary.loc[mask_h, 'high_max'] - trace_summary.loc[mask_h, 'P95'])
    / trace_summary.loc[mask_h, 'P95'].abs().replace({0:1e-9})
)

trace_summary['low_severity_pct'] = 0.0
mask_l = trace_summary['low_anomaly_count'] > 0
trace_summary.loc[mask_l, 'low_severity_pct'] = (
    (trace_summary.loc[mask_l, 'P05'] - trace_summary.loc[mask_l, 'low_min'])
    / trace_summary.loc[mask_l, 'P05'].abs().replace({0:1e-9})
)

# 4) prepare compact top-20 table sorted by max anomaly count
trace_summary['max_anom_count'] = trace_summary[['high_anomaly_count','low_anomaly_count']].max(axis=1)
out_cols = [
    'cmdb_id','trace_name','full_count','P95','P05','window_count',
    'high_anomaly_count','high_first_ts','high_max','high_severity_pct',
    'low_anomaly_count','low_first_ts','low_min','low_severity_pct'
]
top20_traces = (
    trace_summary.sort_values(by='max_anom_count', ascending=False)
    .loc[:, out_cols]
    .head(20)
    .reset_index(drop=True)
)

# 5) total number of distinct trace series scanned
total_trace_series = int(trace_thresholds.shape[0])

# short list (up to 10) with high_anomaly_count >=1 and high_severity_pct > 0
trace_candidates = (
    trace_summary[(trace_summary['high_anomaly_count'] >= 1) & (trace_summary['high_severity_pct'] > 0)]
    .sort_values(by='high_severity_pct', ascending=False)
    .loc[:, ['cmdb_id','trace_name','high_anomaly_count','high_first_ts','high_max','high_severity_pct']]
    .head(10)
    .reset_index(drop=True)
)

# store window-flagged rows for next steps
trace_window_flagged = trace_window.copy()   # includes per-row P95/P05 and flags

# Display compact results: top20_traces, total_trace_series, trace_candidates
top20_traces, total_trace_series, trace_candidates

```
Out[5]:
```
Summary (plain English):

- Scope: scanned 228 distinct (cmdb_id, trace_name) trace series and computed global P95/P05, then checked the incident window (2021-03-25 22:00 → 22:30 +08) for anomalies.

- Top-20 trace series by anomaly counts (compact highlights):
  - Many top entries are trace.self.duration_p95 for containers/hosts (dockerA2, dockerA1, Tomcat02/03/04, dockerB1/B2) showing a few high points (high_anomaly_count typically 2–4) and multiple low points. Example: dockerA2 trace.self.duration_p95 — full_count 31, P95 0.007725, window_count 31, 2 high points (first at 2021-03-25T22:10:00+08) with high_max 0.009 and modest severity (~0.165).
  - Several row_count and duration_* traces between services and MG nodes (e.g., trace.to_MG02.row_count, trace.from_MG02.duration_mean) appear with 2–3 high points and non-zero severity.
  - Many of these top series have full_count near 26–31 (good coverage in the window).

- Candidate trace anomalies to inspect further (high_anomaly_count >=1 and high_severity_pct > 0; short list up to 10):
  1. dockerA2 — trace.self.duration_mean: 2 high points, first at 2021-03-25T22:01:00+08, high_max 0.125633, high_severity_pct ≈ 16.75
  2. dockerA2 — trace.from_MG01.duration_mean: 2 high points, first 22:01, high_max 0.891886, severity ≈ 11.53
  3. MG01 — trace.to_dockerA2.duration_mean: same pattern as (2)
  4. dockerB2 — trace.self.duration_mean: 2 high points, high_max 0.212104, severity ≈ 7.06
  5. Tomcat01 — trace.to_MG01.duration_mean: 2 high points, high_max 0.98839, severity ≈ 6.83
  6. MG01 — trace.from_Tomcat01.duration_mean: same pattern as (5)
  7. dockerB2 — trace.from_MG01.duration_mean: 2 high points, high_max 1.92505, severity ≈ 5.51
  8. MG01 — trace.to_dockerB2.duration_mean: same pattern as (7)
  9. MG02 — trace.from_Tomcat02.duration_mean: 2 high points (first 22:04), high_max 1.027, severity ≈ 5.35
 10. Tomcat02 — trace.to_MG02.duration_mean: 2 high points (first 22:04), high_max 1.027, severity ≈ 5.35

  - Note: most candidate trace anomalies have their earliest high points clustered early in the window (around 22:01–22:10 +08).

- Recommended next steps:
  1. Prioritize the short candidate list above (especially entries with high_severity_pct >> 1) and inspect the raw trace spans around the reported first timestamps (22:01–22:10 +08).
  2. Correlate these trace anomalies with the metric anomalies previously found (ServiceTest/Tomcat/Redis/Mysql candidates) and with logs around the same timestamps.
  3. Use the stored kernel variables for follow-up:
     - trace_thresholds (global P95/P05/full_count per series)
     - trace_window_flagged (window rows with per-row flags and P95/P05)
     to pull raw flagged samples and full context for each candidate.

If you want, I can now extract the raw flagged trace rows (timestamps and values) for any of the candidate series (e.g., dockerA2 trace.self.duration_mean or Tomcat01 trace.to_MG01.duration_mean).

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                        trace_name  full_count        P95        P05  window_count  high_anomaly_count              high_first_ts  high_max  high_severity_pct  low_anomaly_count               low_first_ts  low_min  low_severity_pct
0   dockerA2           trace.self.duration_p95          31   0.007725   0.001000            31                   2  2021-03-25T22:10:00+08:00   0.00900           0.165049                 16  2021-03-25T22:01:00+08:00  0.00100          0.000000
1   dockerA1           trace.self.duration_p95          31   0.008000   0.001000            31                   4  2021-03-25T22:01:00+08:00   0.00800           0.000000                 12  2021-03-25T22:02:00+08:00  0.00100          0.000000
2   Tomcat02           trace.self.duration_p95          31   0.032775   0.023000            31                   2  2021-03-25T22:10:00+08:00   0.03400           0.037376                  8  2021-03-25T22:00:00+08:00  0.02015          0.123913
3   dockerB1           trace.self.duration_p95          31   0.011500   0.009000            31                   2  2021-03-25T22:27:00+08:00   0.01300           0.130435                  7  2021-03-25T22:02:00+08:00  0.00665          0.261111
4   Tomcat03           trace.self.duration_p95          31   0.032000   0.022000            31                   2  2021-03-25T22:19:00+08:00   0.03565           0.114063                  5  2021-03-25T22:00:00+08:00  0.02100          0.045455
5   dockerB2           trace.self.duration_p95          31   0.014850   0.008000            31                   2  2021-03-25T22:28:00+08:00   0.02220           0.494949                  4  2021-03-25T22:03:00+08:00  0.00000          1.000000
6   Tomcat04           trace.self.duration_p95          31   0.031075   0.022000            31                   2  2021-03-25T22:11:00+08:00   0.03300           0.061947                  4  2021-03-25T22:04:00+08:00  0.01650          0.250000
7       MG02     trace.from_Tomcat01.row_count          29  80.200000   1.000000            29                   2  2021-03-25T22:00:00+08:00  87.00000           0.084788                  3  2021-03-25T22:02:00+08:00  1.00000          0.000000
8   dockerA2           trace.to_MG02.row_count          26  16.750000   3.000000            26                   2  2021-03-25T22:00:00+08:00  25.00000           0.492537                  3  2021-03-25T22:01:00+08:00  1.00000          0.666667
9   dockerB1        trace.to_MG02.duration_p95          27   0.091045   0.009000            27                   2  2021-03-25T22:00:00+08:00   0.11105           0.219727                  3  2021-03-25T22:06:00+08:00  0.00900          0.000000
10  dockerA1         trace.from_MG02.row_count          26  50.000000  21.500000            26                   3  2021-03-25T22:10:00+08:00  52.00000           0.040000                  2  2021-03-25T22:06:00+08:00  1.00000          0.953488
11  dockerB2     trace.from_MG02.duration_mean          28   0.355781   0.003000            28                   2  2021-03-25T22:07:00+08:00   0.45100           0.267634                  3  2021-03-25T22:01:00+08:00  0.00100          0.666667
12      MG02   trace.to_dockerB2.duration_mean          28   0.355781   0.003000            28                   2  2021-03-25T22:07:00+08:00   0.45100           0.267634                  3  2021-03-25T22:01:00+08:00  0.00100          0.666667
13      MG02     trace.from_Tomcat02.row_count          28  85.300000   1.000000            28                   2  2021-03-25T22:14:00+08:00  90.00000           0.055100                  3  2021-03-25T22:01:00+08:00  1.00000          0.000000
14  Tomcat02           trace.to_MG02.row_count          28  85.300000   1.000000            28                   2  2021-03-25T22:14:00+08:00  90.00000           0.055100                  3  2021-03-25T22:01:00+08:00  1.00000          0.000000
15  Tomcat01           trace.to_MG02.row_count          29  80.200000   1.000000            29                   2  2021-03-25T22:00:00+08:00  87.00000           0.084788                  3  2021-03-25T22:02:00+08:00  1.00000          0.000000
16      MG02       trace.to_dockerA1.row_count          26  50.000000  21.500000            26                   3  2021-03-25T22:10:00+08:00  52.00000           0.040000                  2  2021-03-25T22:06:00+08:00  1.00000          0.953488
17      MG02     trace.from_dockerA2.row_count          26  16.750000   3.000000            26                   2  2021-03-25T22:00:00+08:00  25.00000           0.492537                  3  2021-03-25T22:01:00+08:00  1.00000          0.666667
18      MG02  trace.from_dockerB1.duration_p95          27   0.091045   0.009000            27                   2  2021-03-25T22:00:00+08:00   0.11105           0.219727                  3  2021-03-25T22:06:00+08:00  0.00900          0.000000
19      IG01    trace.to_Tomcat01.duration_p95          31   1.140325   0.062925            31                   2  2021-03-25T22:10:00+08:00   1.23700           0.084778                  2  2021-03-25T22:12:00+08:00  0.01200          0.809297, 228,     cmdb_id                         trace_name  high_anomaly_count              high_first_ts  high_max  high_severity_pct
0  dockerA2           trace.self.duration_mean                   2  2021-03-25T22:01:00+08:00  0.125633          16.747281
1  dockerA2      trace.from_MG01.duration_mean                   2  2021-03-25T22:01:00+08:00  0.891886          11.525785
2      MG01    trace.to_dockerA2.duration_mean                   2  2021-03-25T22:01:00+08:00  0.891886          11.525785
3  dockerB2           trace.self.duration_mean                   2  2021-03-25T22:01:00+08:00  0.212104           7.055143
4  Tomcat01        trace.to_MG01.duration_mean                   2  2021-03-25T22:01:00+08:00  0.988390           6.826755
5      MG01  trace.from_Tomcat01.duration_mean                   2  2021-03-25T22:01:00+08:00  0.988390           6.826755
6  dockerB2      trace.from_MG01.duration_mean                   2  2021-03-25T22:01:00+08:00  1.925052           5.511915
7      MG01    trace.to_dockerB2.duration_mean                   2  2021-03-25T22:01:00+08:00  1.925052           5.511915
8      MG02  trace.from_Tomcat02.duration_mean                   2  2021-03-25T22:04:00+08:00  1.027000           5.349238
9  Tomcat02        trace.to_MG02.duration_mean                   2  2021-03-25T22:04:00+08:00  1.027000           5.349238)```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

# reuse kernel variables: summary (per-series), window_flagged_rows, thresholds_per_series
tz = pytz.timezone('Asia/Shanghai')

# Allowed components
allowed = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
           'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']

# Work on a copy of the per-series summary (already contains needed columns)
s = summary.copy()

# Restrict to allowed components
s_allowed = s[s['cmdb_id'].isin(allowed)].copy()

# Define flagged series in window
s_allowed['is_flagged'] = (s_allowed['high_anomaly_count'] > 0) | (s_allowed['low_anomaly_count'] > 0)

# Compute per-series earliest anomaly timestamp (taking min of high_first_ts and low_first_ts)
def parse_iso_to_dt(val):
    if pd.isna(val) or val is None:
        return pd.NaT
    # parse ISO (may include timezone); produce tz-aware in Asia/Shanghai
    try:
        return pd.to_datetime(val, utc=True).tz_convert(tz)
    except Exception:
        return pd.NaT

s_allowed['high_first_dt'] = s_allowed['high_first_ts'].apply(parse_iso_to_dt)
s_allowed['low_first_dt']  = s_allowed['low_first_ts'].apply(parse_iso_to_dt)
s_allowed['series_earliest_dt'] = s_allowed[['high_first_dt','low_first_dt']].min(axis=1)

# Series-level max severity
s_allowed['series_max_sev'] = s_allowed[['high_severity_pct','low_severity_pct']].max(axis=1).fillna(0.0)

# Aggregate per component only using flagged series
flagged = s_allowed[s_allowed['is_flagged']].copy()

# If no flagged series, produce empty outputs gracefully
if flagged.shape[0] == 0:
    component_summary_full = pd.DataFrame(columns=[
        'component','earliest_anom_ts','flagged_series_count','max_severity','total_window_points'
    ])
    component_summary_top20 = component_summary_full.copy()
    num_components_with_flags = 0
else:
    # Group by component
    grouped = flagged.groupby('cmdb_id', sort=True)

    rows = []
    for comp, grp in grouped:
        # earliest_anom_ts = min of series_earliest_dt (ignore NaT)
        earliest_dt = grp['series_earliest_dt'].min()
        # flagged_series_count
        flagged_series_count = int(grp.shape[0])
        # max_severity across series
        max_severity = float(grp['series_max_sev'].max())
        # total_window_points sum of window_count across flagged series
        total_window_points = int(grp['window_count'].sum())
        rows.append({
            'component': comp,
            'earliest_anom_ts_dt': earliest_dt,      # keep dt for sorting
            'flagged_series_count': flagged_series_count,
            'max_severity': max_severity,
            'total_window_points': total_window_points
        })

    component_summary_full = pd.DataFrame(rows)

    # Format earliest_anom_ts as ISO strings in Asia/Shanghai (None -> None)
    def dt_to_iso_or_none(dtval):
        if pd.isna(dtval):
            return None
        return pd.to_datetime(dtval).tz_convert(tz).isoformat()

    component_summary_full['earliest_anom_ts'] = component_summary_full['earliest_anom_ts_dt'].apply(dt_to_iso_or_none)

    # Sort by earliest_anom_ts ascending (NaT last), then max_severity descending
    component_summary_full = component_summary_full.sort_values(
        by=['earliest_anom_ts_dt','max_severity'],
        ascending=[True, False],
        na_position='last'
    ).reset_index(drop=True)

    # Prepare top-20 display (select columns and limit)
    component_summary_top20 = component_summary_full.loc[:, [
        'component','earliest_anom_ts','flagged_series_count','max_severity','total_window_points'
    ]].head(20)

    num_components_with_flags = int(component_summary_full.shape[0])

# Display compact results: top-20 table and number of components with flags
component_summary_top20, num_components_with_flags

```
Out[6]:
```
Summary (plain English):

- Components with flagged metric series in the incident window: 14 (apache01, apache02, Tomcat01–04, MG01–02, IG01–02, Mysql01–02, Redis01–02).

- Earliest detected anomaly time for all listed components: 2021-03-25T22:00:00+08:00.

- Per-component summary (compact highlights, sorted by earliest then severity):
  1. Mysql01 — flagged_series_count 224, max_severity 1.136e+12, total_window_points 4875
  2. MG01    — flagged_series_count 104, max_severity 1.0e+9, total_window_points 1556
  3. Mysql02 — flagged_series_count 224, max_severity 1.0e+9, total_window_points 4834
  4. Tomcat03— flagged_series_count 122, max_severity 1.0e+9, total_window_points 2010
  5. Tomcat01— flagged_series_count 120, max_severity ~8.85,    total_window_points 1950
  6. Redis01 — flagged_series_count 114, max_severity ~5.67,    total_window_points 1563
  7. Redis02 — flagged_series_count 110, max_severity 3.00,     total_window_points 1916
  8. Tomcat02— flagged_series_count 122, max_severity ~2.64,    total_window_points 2024
  9. apache02— flagged_series_count 82,  max_severity ~2.08,    total_window_points 922
 10. MG02    — flagged_series_count 106, max_severity ~1.75,    total_window_points 1582
 11. Tomcat04— flagged_series_count 123, max_severity ~1.26,    total_window_points 2000
 12. IG01    — flagged_series_count 90,  max_severity 1.00,     total_window_points 1096
 13. IG02    — flagged_series_count 88,  max_severity 1.00,     total_window_points 1107
 14. apache01— flagged_series_count 81,  max_severity ~0.99,    total_window_points 879

- Notes and interpretation:
  - The very large max_severity values for many MySQL/MG/Tomcat entries (e.g., 1.136e+12, 1.0e+9) are likely due to historical P95 = 0 for those particular KPIs so any positive window value produces an extremely large ratio. These are probably threshold-artifacts and should be inspected before treating them as real “severity.”
  - Tomcat01 shows a meaningful non-artifact max_severity (~8.85) and Redis/Service metrics show moderate real severity (e.g., Redis01 ~5.67). These are stronger candidates for real impact.
  - All components show earliest anomaly at the start of the window (22:00+08), so the incident appears to affect many components concurrently or the metric collection baseline changed at that time.

- Recommended next actions:
  1. Inspect MySQL and MG series where P95==0 to confirm whether the extreme severity is an artifact (look at raw values and historical distribution).
  2. Prioritize inspection of Tomcat01 and Redis series with sizable non-artifact severity (check raw flagged samples, thread/disk/CPU metrics and related logs around 22:00–22:10+08).
  3. Correlate these component-level findings with trace anomalies and application logs around the earliest anomaly timestamps to identify likely root causes.

If you want, I can now pull the raw flagged metric samples for the top components (e.g., Tomcat01, Redis01, Mysql01) for direct inspection.

The original code execution output of IPython Kernel is also provided below for reference:

(   component           earliest_anom_ts  flagged_series_count  max_severity  total_window_points
0    Mysql01  2021-03-25T22:00:00+08:00                   224  1.136000e+12                 4875
1       MG01  2021-03-25T22:00:00+08:00                   104  1.000000e+09                 1556
2    Mysql02  2021-03-25T22:00:00+08:00                   224  1.000000e+09                 4834
3   Tomcat03  2021-03-25T22:00:00+08:00                   122  1.000000e+09                 2010
4   Tomcat01  2021-03-25T22:00:00+08:00                   120  8.848427e+00                 1950
5    Redis01  2021-03-25T22:00:00+08:00                   114  5.666667e+00                 1563
6    Redis02  2021-03-25T22:00:00+08:00                   110  3.000000e+00                 1916
7   Tomcat02  2021-03-25T22:00:00+08:00                   122  2.636364e+00                 2024
8   apache02  2021-03-25T22:00:00+08:00                    82  2.076923e+00                  922
9       MG02  2021-03-25T22:00:00+08:00                   106  1.751796e+00                 1582
10  Tomcat04  2021-03-25T22:00:00+08:00                   123  1.264309e+00                 2000
11      IG01  2021-03-25T22:00:00+08:00                    90  1.000000e+00                 1096
12      IG02  2021-03-25T22:00:00+08:00                    88  1.000000e+00                 1107
13  apache01  2021-03-25T22:00:00+08:00                    81  9.894158e-01                  879, 14)```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse trace_window_flagged from kernel (contains window rows with P95/P05 and flags)
# If not present, fall back to trace_window_flagged name used earlier
try:
    tw = trace_window_flagged
except NameError:
    tw = trace_window_flagged  # will raise if truly missing

# Ensure timestamp is integer
tw = tw.copy()
tw['timestamp'] = tw['timestamp'].astype(int)

# helper to add timestamp_dt in Asia/Shanghai
def add_timestamp_dt(df):
    if 'timestamp_dt' not in df.columns:
        df = df.copy()
        df['timestamp_dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S%z')
    return df

# Target series list (cmdb_id, trace_name)
targets = [
    ('Tomcat01', 'trace.to_MG01.duration_mean'),
    ('MG01',      'trace.from_Tomcat01.duration_mean'),
    ('Tomcat02', 'trace.to_MG02.duration_mean'),
    ('MG02',      'trace.from_Tomcat02.duration_mean')
]

# Containers to store per-series full extracted flagged rows
flagged_series_dfs = {}
summary_rows = []

for cmdb_id, trace_name in targets:
    mask = (tw['cmdb_id'] == cmdb_id) & (tw['trace_name'] == trace_name) & (tw['high_flag'] == True)
    df_flagged = tw.loc[mask, ['timestamp','cmdb_id','trace_name','value']].copy()
    df_flagged = add_timestamp_dt(df_flagged).sort_values('timestamp', ascending=True).reset_index(drop=True)
    # store full extracted rows in kernel variable mapping
    key = f"{cmdb_id}__{trace_name}"
    flagged_series_dfs[key] = df_flagged
    # compute count and earliest high ts
    high_count = int(df_flagged.shape[0])
    earliest_high_ts = None
    if high_count > 0:
        earliest_high_ts = pd.to_datetime(int(df_flagged['timestamp'].min()), unit='s', utc=True).tz_convert(tz).isoformat()
    summary_rows.append({
        'cmdb_id': cmdb_id,
        'trace_name': trace_name,
        'high_count': high_count,
        'earliest_high_ts': earliest_high_ts,
        'top_rows_shown': min(20, high_count)
    })

# Build summary DataFrame
summary_df = pd.DataFrame(summary_rows)

# Build top-rows display for each series limited to 20 rows
top_rows_display = {
    key: df.head(20).loc[:, ['timestamp','timestamp_dt','cmdb_id','trace_name','value']]
    for key, df in flagged_series_dfs.items()
}

# Also store the full concatenated extracted rows in a kernel variable for follow-up
full_extracted_traces = pd.concat(
    [df.assign(series_key=key) for key, df in flagged_series_dfs.items()],
    ignore_index=True
) if len(flagged_series_dfs) > 0 else pd.DataFrame(columns=['timestamp','timestamp_dt','cmdb_id','trace_name','value','series_key'])

# Display compact results: summary table and the top flagged rows per target (up to 20 each)
summary_df, top_rows_display['Tomcat01__trace.to_MG01.duration_mean'], top_rows_display['MG01__trace.from_Tomcat01.duration_mean'], top_rows_display['Tomcat02__trace.to_MG02.duration_mean'], top_rows_display['MG02__trace.from_Tomcat02.duration_mean']

```
Out[7]:
```
Summary (plain English):

- Four requested trace series had flagged high points in the incident window; each has 2 flagged high samples.

- Per-series details:
  1. Tomcat01 — trace.to_MG01.duration_mean
     - flagged high count: 2
     - earliest high: 2021-03-25T22:01:00+08:00
     - flagged samples (shown up to 20):
       - 2021-03-25 22:01:00+08 | value = 0.988390
       - 2021-03-25 22:13:00+08 | value = 0.135047
  2. MG01 — trace.from_Tomcat01.duration_mean
     - flagged high count: 2
     - earliest high: 2021-03-25T22:01:00+08:00
     - flagged samples:
       - 2021-03-25 22:01:00+08 | value = 0.988390
       - 2021-03-25 22:13:00+08 | value = 0.135047
  3. Tomcat02 — trace.to_MG02.duration_mean
     - flagged high count: 2
     - earliest high: 2021-03-25T22:04:00+08:00
     - flagged samples:
       - 2021-03-25 22:04:00+08 | value = 1.027000
       - 2021-03-25 22:13:00+08 | value = 0.165524
  4. MG02 — trace.from_Tomcat02.duration_mean
     - flagged high count: 2
     - earliest high: 2021-03-25T22:04:00+08:00
     - flagged samples:
       - 2021-03-25 22:04:00+08 | value = 1.027000
       - 2021-03-25 22:13:00+08 | value = 0.165524

- Stored variables for follow-up:
  - full_extracted_traces: full concatenation of extracted flagged rows for these series (all flagged rows stored).
  - flagged_series_dfs: per-series full DataFrame mapping (keys like "Tomcat01__trace.to_MG01.duration_mean").

If you want, I can now show the full extracted rows for any one of these series or correlate these timestamps with metric anomalies.

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id                         trace_name  high_count           earliest_high_ts  top_rows_shown
0  Tomcat01        trace.to_MG01.duration_mean           2  2021-03-25T22:01:00+08:00               2
1      MG01  trace.from_Tomcat01.duration_mean           2  2021-03-25T22:01:00+08:00               2
2  Tomcat02        trace.to_MG02.duration_mean           2  2021-03-25T22:04:00+08:00               2
3      MG02  trace.from_Tomcat02.duration_mean           2  2021-03-25T22:04:00+08:00               2,     timestamp              timestamp_dt   cmdb_id                   trace_name     value
0  1616680860  2021-03-25 22:01:00+0800  Tomcat01  trace.to_MG01.duration_mean  0.988390
1  1616681580  2021-03-25 22:13:00+0800  Tomcat01  trace.to_MG01.duration_mean  0.135047,     timestamp              timestamp_dt cmdb_id                         trace_name     value
0  1616680860  2021-03-25 22:01:00+0800    MG01  trace.from_Tomcat01.duration_mean  0.988390
1  1616681580  2021-03-25 22:13:00+0800    MG01  trace.from_Tomcat01.duration_mean  0.135047,     timestamp              timestamp_dt   cmdb_id                   trace_name     value
0  1616681040  2021-03-25 22:04:00+0800  Tomcat02  trace.to_MG02.duration_mean  1.027000
1  1616681580  2021-03-25 22:13:00+0800  Tomcat02  trace.to_MG02.duration_mean  0.165524,     timestamp              timestamp_dt cmdb_id                         trace_name     value
0  1616681040  2021-03-25 22:04:00+0800    MG02  trace.from_Tomcat02.duration_mean  1.027000
1  1616681580  2021-03-25 22:13:00+0800    MG02  trace.from_Tomcat02.duration_mean  0.165524)```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz
import re

tz = pytz.timezone('Asia/Shanghai')

# Reuse kernel DataFrames: metric_df, log_df, thresholds_per_series, window_flagged_rows if needed
# Incident window
start_ts = 1616680800
end_ts   = 1616682600

# 1) Metrics for Tomcat01 with requested KPI patterns
# pattern definitions (case-sensitive as KPIs appear case-specific)
patterns = [
    r'OSLinux-CPU_CPU_CPUWio',
    r'LOCALDISK.*sda.*DSKRead',
    r'LOCALDISK.*sda.*DSKRTps',
    r'OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT',
    r'Tomcat-Threads',       # contains Tomcat-Threads
    r'ThreadCount',         # contains ThreadCount
    r'CurrentThreads'       # contains CurrentThreads
]
pattern_regex = '(' + ')|('.join(patterns) + ')'

# Filter metric rows to Tomcat01 and window
m_tomcat = metric_df[
    (metric_df['cmdb_id'] == 'Tomcat01') &
    (metric_df['timestamp'] >= start_ts) &
    (metric_df['timestamp'] <= end_ts) &
    (metric_df['kpi_name'].str.contains(pattern_regex, regex=True, na=False))
].copy()

# Add timestamp_dt in Asia/Shanghai
m_tomcat['timestamp_dt'] = pd.to_datetime(m_tomcat['timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S%z')

# Store full extracted subset in kernel variable
extracted_metrics_tomcat01 = m_tomcat.copy()

# Merge global thresholds (P95) from thresholds_per_series (ensure it exists)
# thresholds_per_series has columns: cmdb_id,kpi_name,P95,P05,full_count (as created earlier)
thr = thresholds_per_series.copy()
thr_subset = thr[thr['cmdb_id']=='Tomcat01'].loc[:, ['cmdb_id','kpi_name','P95','P05','full_count']]

m_tomcat = m_tomcat.merge(thr_subset, on=['cmdb_id','kpi_name'], how='left')

# Flag rows with value >= global P95 (rule: P95 computed globally before filtering)
m_tomcat['high_flag'] = m_tomcat['value'] >= m_tomcat['P95']

# Per-KPI aggregates requested:
# - total rows in window (window_count)
# - max value (window_max)
# - min value (window_min)
# - earliest timestamp of max (earliest_max_ts)
# - count of values >= global P95 (count_ge_P95)
agg_count = m_tomcat.groupby('kpi_name', sort=True).agg(
    window_count = ('value','count'),
    window_max = ('value','max'),
    window_min = ('value','min')
).reset_index()

# earliest timestamp of max: join back to find rows equal to window_max then take min timestamp
agg_max_ts = (
    m_tomcat.loc[:, ['kpi_name','timestamp','value']]
    .merge(agg_count[['kpi_name','window_max']], on='kpi_name', how='left')
)
agg_max_ts = agg_max_ts[agg_max_ts['value'] == agg_max_ts['window_max']].groupby('kpi_name', sort=True)['timestamp'].min().reset_index().rename(columns={'timestamp':'earliest_max_ts'})

# count values >= P95
agg_ge_p95 = (
    m_tomcat[m_tomcat['high_flag']]
    .groupby('kpi_name', sort=True)['value']
    .count()
    .reset_index()
    .rename(columns={'value':'count_ge_P95'})
)

# Merge aggregates
agg_metrics = agg_count.merge(agg_max_ts, on='kpi_name', how='left').merge(agg_ge_p95, on='kpi_name', how='left')
agg_metrics['count_ge_P95'] = agg_metrics['count_ge_P95'].fillna(0).astype(int)

# Convert earliest_max_ts to ISO tz string
def ts_to_iso(v):
    if pd.isna(v):
        return None
    return pd.to_datetime(int(v), unit='s', utc=True).tz_convert(tz).isoformat()

agg_metrics['earliest_max_ts'] = agg_metrics['earliest_max_ts'].apply(lambda x: ts_to_iso(x))
# keep columns in requested order
agg_metrics = agg_metrics.loc[:, ['kpi_name','window_count','window_max','window_min','earliest_max_ts','count_ge_P95']]

# Prepare raw rows display limited to 50 rows total (user requested)
raw_metric_head = extracted_metrics_tomcat01.loc[:, ['timestamp','timestamp_dt','cmdb_id','kpi_name','value']].sort_values(['kpi_name','timestamp']).head(50)

# 2) Logs for Tomcat01 in the same window
logs_tomcat = log_df[
    (log_df['cmdb_id'] == 'Tomcat01') &
    (log_df['timestamp'] >= start_ts) &
    (log_df['timestamp'] <= end_ts)
].copy()

# add timestamp_dt
logs_tomcat['timestamp_dt'] = pd.to_datetime(logs_tomcat['timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S%z')

# Store full extracted logs in kernel var
extracted_logs_tomcat01 = logs_tomcat.copy()

# compute count of log.error_count > 0 and earliest timestamp when that occurs
mask_error_count = (logs_tomcat['log_name'] == 'log.error_count') & (logs_tomcat['value'] > 0)
log_error_count = int(mask_error_count.sum())
if log_error_count > 0:
    earliest_error_ts = pd.to_datetime(int(logs_tomcat.loc[mask_error_count, 'timestamp'].min()), unit='s', utc=True).tz_convert(tz).isoformat()
else:
    earliest_error_ts = None

# Limit displayed log rows to 50
logs_head = logs_tomcat.loc[:, ['timestamp','timestamp_dt','cmdb_id','log_name','value']].sort_values('timestamp').head(50)

# Final compact outputs (as variables for IPython display)
agg_metrics, raw_metric_head, logs_head, log_error_count, earliest_error_ts, extracted_metrics_tomcat01.shape, extracted_logs_tomcat01.shape

```
Out[8]:
```
Summary (plain English):

- Extraction scope: Tomcat01 metrics and logs in the incident window [2021-03-25 22:00:00+08 — 22:30:00+08].

- Metrics (per-KPI aggregates, only KPIs that matched requested patterns):
  - OSLinux-CPU_CPU_CPUWio
    - rows in window: 31, max=1.8833, min=0.012471
    - earliest max: 2021-03-25T22:22:00+08
    - count >= global P95: 2
  - OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps
    - rows: 16, max=6.2000, min=0.0000
    - earliest max: 2021-03-25T22:22:00+08
    - count >= P95: 1
  - OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead
    - rows: 16, max=406.9333, min=0.0000
    - earliest max: 2021-03-25T22:22:00+08
    - count >= P95: 1
  - Another DSKRead-like KPI (long name)
    - rows: 11, max=470.0000, min=7.0000
    - earliest max: 2021-03-25T22:23:00+08
    - count >= P95: 1
  - OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT
    - rows: 30, max=2, min=0
    - earliest max: 2021-03-25T22:01:00+08
    - count >= P95: 2
  - Tomcat thread metrics (CurrentThreads / ThreadCount / MaxThreads variants)
    - one thread KPI: rows 31, max=100, min=100, earliest max=2021-03-25T22:00:00+08, count >= P95 = 31 (all samples)
    - another thread KPI: rows 31, max=2, min=0, earliest max=2021-03-25T22:13:00+08, count >= P95 = 9
    - MaxThreads KPI: rows 31, max=500, min=500, count >= P95 = 31

- Raw metric rows returned: a selected sample (up to 50 rows shown). Full extracted metric subset size: 197 rows (stored).

- Logs for Tomcat01:
  - Extracted rows shown (up to 50). Full extracted logs size: 68 rows (stored).
  - Count of log.error_count > 0 occurrences in window: 0
  - Earliest log.error_count > 0: none

- Key observations / suggested follow-up:
  - Thread-related KPIs show many values equal to their max (e.g., CurrentThreads/MaxThreads with all 31 samples flagged >= P95). This may indicate sustained thread counts or a threshold artifact (verify global P95 values).
  - Disk read peaks (DSKRead) and DSKRTps spikes occur around 22:22–22:23 and have single-point P95 breaches — worth inspecting raw samples and the process behavior at that time.
  - CPU WIO shows 2 high points (earliest around 22:22) — correlate with disk spikes.
  - TCP-FIN-WAIT saw small spikes earlier (first at 22:01) — may reflect connection state changes but needs correlation with application traces/logs.
  - No log.error_count > 0 events were found for Tomcat01 in the window.

- Kernel variables saved for follow-up:
  - extracted_metrics_tomcat01 (full metric subset for Tomcat01 matching patterns)
  - extracted_logs_tomcat01 (full log rows for Tomcat01)
  - window_flagged_rows / thresholds_per_series remain available for further correlation.

If you want, I can: show the full extracted metric rows for a specific KPI (e.g., DSKRead or CPU WIO), or correlate the metric spike timestamps (22:22–22:23) with the trace anomalies previously extracted.

The original code execution output of IPython Kernel is also provided below for reference:

(                                            kpi_name  window_count  window_max  window_min            earliest_max_ts  count_ge_P95
0                             OSLinux-CPU_CPU_CPUWio            31      1.8833    0.012471  2021-03-25T22:22:00+08:00             2
1    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps            16      6.2000    0.000000  2021-03-25T22:22:00+08:00             1
2    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead            16    406.9333    0.000000  2021-03-25T22:22:00+08:00             1
3  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...            11    470.0000    7.000000  2021-03-25T22:23:00+08:00             1
4       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT            30      2.0000    0.000000  2021-03-25T22:01:00+08:00             2
5  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...            31    100.0000  100.000000  2021-03-25T22:00:00+08:00            31
6  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...            31      2.0000    0.000000  2021-03-25T22:13:00+08:00             9
7  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...            31    500.0000  500.000000  2021-03-25T22:00:00+08:00            31,         timestamp              timestamp_dt   cmdb_id                                         kpi_name     value
687    1616680800  2021-03-25 22:00:00+0800  Tomcat01                           OSLinux-CPU_CPU_CPUWio  0.020782
1724   1616680860  2021-03-25 22:01:00+0800  Tomcat01                           OSLinux-CPU_CPU_CPUWio  0.212000
2650   1616680920  2021-03-25 22:02:00+0800  Tomcat01                           OSLinux-CPU_CPU_CPUWio  0.153800
3698   1616680980  2021-03-25 22:03:00+0800  Tomcat01                           OSLinux-CPU_CPU_CPUWio  0.029104
4586   1616681040  2021-03-25 22:04:00+0800  Tomcat01                           OSLinux-CPU_CPU_CPUWio  0.029100
...           ...                       ...       ...                                              ...       ...
27742  1616682480  2021-03-25 22:28:00+0800  Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps  0.000000
29487  1616682600  2021-03-25 22:30:00+0800  Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps  0.183333
703    1616680800  2021-03-25 22:00:00+0800  Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead  0.000000
2692   1616680920  2021-03-25 22:02:00+0800  Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead  0.000000
4629   1616681040  2021-03-25 22:04:00+0800  Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead  0.000000

[50 rows x 5 columns],       timestamp              timestamp_dt   cmdb_id          log_name   value
0    1616680800  2021-03-25 22:00:00+0800  Tomcat01   log.error_count    0.00
1    1616680800  2021-03-25 22:00:00+0800  Tomcat01     log.row_count  161.00
13   1616680860  2021-03-25 22:01:00+0800  Tomcat01   log.error_count    0.00
14   1616680860  2021-03-25 22:01:00+0800  Tomcat01  log.gc_pause_max    0.06
15   1616680860  2021-03-25 22:01:00+0800  Tomcat01     log.row_count  152.00
..          ...                       ...       ...               ...     ...
258  1616682000  2021-03-25 22:20:00+0800  Tomcat01     log.row_count  109.00
270  1616682060  2021-03-25 22:21:00+0800  Tomcat01   log.error_count    0.00
271  1616682060  2021-03-25 22:21:00+0800  Tomcat01  log.gc_pause_max    0.03
272  1616682060  2021-03-25 22:21:00+0800  Tomcat01     log.row_count  119.00
283  1616682120  2021-03-25 22:22:00+0800  Tomcat01   log.error_count    0.00

[50 rows x 5 columns], 0, None, (197, 5), (68, 5))

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse kernel variables:
# - summary (per-series thresholds + window stats + severity)
# - window_merged (window rows with P95/P05 and high_flag/low_flag)
# - thresholds_per_series (global thresholds)
# These were created in earlier steps.

# 1) Restrict to MG01 series and apply selection condition
mg = summary.copy()
mg01_all = mg[mg['cmdb_id'] == 'MG01'].copy()

# Ensure numeric fields present
for col in ['P95','window_max','high_severity_pct','low_severity_pct']:
    if col not in mg01_all.columns:
        mg01_all[col] = np.nan
mg01_all['window_max'] = mg01_all.get('window_max', np.nan).astype(float)
mg01_all['P95'] = mg01_all.get('P95', np.nan).astype(float)
mg01_all['high_severity_pct'] = mg01_all.get('high_severity_pct', 0.0).fillna(0.0).astype(float)
mg01_all['low_severity_pct'] = mg01_all.get('low_severity_pct', 0.0).fillna(0.0).astype(float)

# Selection condition per user
cond = (
    (mg01_all['high_severity_pct'] > 0.0) |
    (mg01_all['low_severity_pct'] > 0.0) |
    ((mg01_all['P95'] == 0.0) & (mg01_all['window_max'] > 0.0))
)

mg01_selected = mg01_all[cond].copy()

# 2) Compute streaks (longest consecutive run of flagged points) using window_merged
# Ensure window_merged exists
wm = window_merged.copy()
wm['timestamp'] = wm['timestamp'].astype(int)

# helper to compute streak length for a boolean flag series in chronological order
def compute_streak_max(timestamps):
    # timestamps: numpy array of epoch seconds of flagged points (sorted ascending)
    if len(timestamps) == 0:
        return 0
    if len(timestamps) == 1:
        return 1
    diffs = np.diff(timestamps)
    consec = diffs == 60  # consecutive minute sampling
    # build group ids
    group_ids = np.empty(len(timestamps), dtype=int)
    group_ids[0] = 0
    group_ids[1:] = (~consec).cumsum()
    counts = np.bincount(group_ids)
    return int(counts.max())

# Prepare maps for quick lookup: group window_merged by (cmdb_id,kpi_name)
wm_groups = wm.groupby(['cmdb_id','kpi_name'], sort=False)

high_streaks = {}
low_streaks = {}

for idx, row in mg01_selected[['cmdb_id','kpi_name']].iterrows():
    cm = row['cmdb_id']
    kp = row['kpi_name']
    key = (cm,kp)
    try:
        grp = wm_groups.get_group(key).sort_values('timestamp', ascending=True)
    except KeyError:
        grp = pd.DataFrame(columns=wm.columns)
    # high_flag timestamps
    high_ts = grp.loc[grp['high_flag'], 'timestamp'].astype(int).values
    low_ts  = grp.loc[grp['low_flag'],  'timestamp'].astype(int).values
    high_streaks[key] = compute_streak_max(high_ts)
    low_streaks[key]  = compute_streak_max(low_ts)

# Attach streaks to mg01_selected
def get_streak(cm,kp, mapping):
    return mapping.get((cm,kp), 0)

mg01_selected['high_streak_max'] = mg01_selected.apply(lambda r: get_streak(r['cmdb_id'], r['kpi_name'], high_streaks), axis=1)
mg01_selected['low_streak_max']  = mg01_selected.apply(lambda r: get_streak(r['cmdb_id'], r['kpi_name'], low_streaks), axis=1)

# 3) Build compact table with requested columns and sort by max severity descending
mg01_selected['series_max_sev'] = mg01_selected[['high_severity_pct','low_severity_pct']].max(axis=1)
cols = [
    'cmdb_id','kpi_name','full_count','P95','P05','window_count',
    'high_anomaly_count','high_first_ts','high_max','high_severity_pct',
    'low_anomaly_count','low_first_ts','low_min','low_severity_pct',
    'high_streak_max','low_streak_max','series_max_sev'
]
# ensure columns exist
for c in cols:
    if c not in mg01_selected.columns:
        mg01_selected[c] = np.nan

mg01_table = (
    mg01_selected
    .loc[:, cols]
    .sort_values(by='series_max_sev', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# 4) Short list (up to 5) where high_streak_max >=2 or low_streak_max >=2
fault_like = mg01_selected[
    (mg01_selected['high_streak_max'] >= 2) | (mg01_selected['low_streak_max'] >= 2)
].copy()

fault_short = fault_like.loc[:, ['cmdb_id','kpi_name','high_streak_max','low_streak_max','series_max_sev']].sort_values(
    by=['high_streak_max','low_streak_max','series_max_sev'], ascending=[False,False,False]
).head(5).reset_index(drop=True)

# 5) For top 5 series by series_max_sev, extract raw flagged rows (high_flag OR low_flag) within window
top5_keys = mg01_selected.sort_values(by='series_max_sev', ascending=False).head(5)[['cmdb_id','kpi_name']].apply(tuple, axis=1).tolist()

mg01_top5_flagged_full = {}    # store full flagged rows per series
mg01_top5_flagged_preview = {} # limited to 20 rows for display

for cm,kp in top5_keys:
    try:
        grp = wm_groups.get_group((cm,kp)).sort_values('timestamp', ascending=True)
    except KeyError:
        grp = pd.DataFrame(columns=wm.columns)
    # select flagged rows (high or low)
    flagged = grp[(grp['high_flag']==True) | (grp['low_flag']==True)].copy()
    if not flagged.empty:
        flagged['timestamp_dt'] = pd.to_datetime(flagged['timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S%z')
        mg01_top5_flagged_full[f"{cm}__{kp}"] = flagged.loc[:, ['timestamp','timestamp_dt','cmdb_id','kpi_name','value','high_flag','low_flag']]
        mg01_top5_flagged_preview[f"{cm}__{kp}"] = mg01_top5_flagged_full[f"{cm}__{kp}"].head(20).loc[:, ['timestamp','timestamp_dt','cmdb_id','kpi_name','value','high_flag','low_flag']]
    else:
        mg01_top5_flagged_full[f"{cm}__{kp}"] = flagged
        mg01_top5_flagged_preview[f"{cm}__{kp}"] = flagged

# Also store full MG01 flagged subset (all MG01 rows with high_flag or low_flag) for follow-up
mg01_flagged_rows_full = wm[(wm['cmdb_id']=='MG01') & ((wm['high_flag']==True) | (wm['low_flag']==True))].copy()
if not mg01_flagged_rows_full.empty:
    mg01_flagged_rows_full['timestamp_dt'] = pd.to_datetime(mg01_flagged_rows_full['timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S%z')

# Final compact outputs to display:
# - mg01_table (top 20 series meeting condition)
# - fault_short (up to 5 likely fault series with streaks)
# - mg01_top5_flagged_preview (dict of up to 20 raw flagged rows per top-5 series)
mg01_table, fault_short, mg01_top5_flagged_preview

```
Out[9]:
```
Summary (plain English):

- Scope: scanned MG01 series and returned those meeting (high_severity_pct>0 OR low_severity_pct>0 OR (P95==0 AND window_max>0)). The compact table shows up to 20 series (displayed in the kernel).

- Key findings (top entries / highlights):
  - OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT
    - full_count 30, window_count 30, high_anomaly_count 30, low_anomaly_count 29
    - P95 = 0, window_max = 1 → produced extremely large reported high_severity_pct (artifact of P95==0)
    - very long runs: high_streak_max = 29, low_streak_max = 14 (sustained states)
  - Several disk I/O metrics (sda/sdb DSKBps, DSKWrite, DSKRead, DSKPer..., DSKRTps) show single-point or small spikes (window_count ~9–15, usually 1 high point).
  - CPU metrics (OSLinux-CPU_CPUWio, CPU CPULoad, CPUSysTime) have small but nonzero severity and a few high points.
  - JVM memory metric (JVM-Memory_7779_JVM_Memory_HeapMemoryUsed) had 2 high points and moderate severity.
  - Many series have P95==0 with occasional nonzero window values — these produce large severity ratios and should be treated as threshold artifacts until validated.

- Fault-like series (high_streak_max >=2 or low_streak_max >=2) — short list (up to 5):
  1. MG01 — OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT (high_streak_max 29, low_streak_max 14)
  2. MG01 — OSLinux-CPU_CPU_CPULoad (low_streak_max 2)
  3. MG01 — OSLinux-OSLinux_ZABBIX_Host_Uptime (low_streak_max 2)
  (these and others with multi-sample streaks are likely sustained conditions worth investigating)

- Raw flagged rows (top-5 series by severity) — previews were extracted and stored (up to 20 rows each). Example previews included:
  - MG01__OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT — many flagged rows across the window (mostly 0.0, with a 1.0 at 22:14)
  - MG01__OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps — one high value 6.0 at 22:19 among many zeros
  - MG01__OSLinux-CPU_CPU_CPUWio — high points at 22:10 (0.507129) and 22:27 (0.7691)
  - MG01__OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps and sdb_DSKWrite — single high rows shown (e.g., sdb_DSKWrite 853.6 at 22:02)

- Data stored in kernel for follow-up:
  - mg01_table (selected per-series summary),
  - fault_short (short list of streaky series),
  - mg01_top5_flagged_full / mg01_top5_flagged_preview (raw flagged rows per top series),
  - mg01_flagged_rows_full (all MG01 flagged rows).

- Recommended next steps:
  1. Inspect raw flagged samples for the TCP-CLOSE-WAIT series — long streaks indicate sustained connection-state conditions; determine if this is application-level behavior or metric/instrumentation artifact.
  2. Treat series with P95 == 0 and nonzero window_max cautiously — confirm historical baseline and whether P95 should be recomputed with more data or different quantile method.
  3. Correlate MG01 disk I/O spikes and CPU WIO high points (22:02 and 22:19–22:23 ranges) with traces/logs and with other components (Tomcat/Mysql) for root-cause investigation.

If you want, I can now:
- show the full flagged rows for any one MG01 series (from mg01_top5_flagged_full), or
- produce correlated timelines between MG01 disk/CPU spikes and Tomcat trace anomalies.

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                                           kpi_name  full_count           P95           P05  window_count  high_anomaly_count              high_first_ts      high_max  high_severity_pct  low_anomaly_count               low_first_ts       low_min  low_severity_pct  high_streak_max  low_streak_max  series_max_sev
0     MG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT          30  0.000000e+00  0.000000e+00            30                  30  2021-03-25T22:00:00+08:00  1.000000e+00       1.000000e+09                 29  2021-03-25T22:00:00+08:00  0.000000e+00          0.000000               29              14    1.000000e+09
1     MG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps          15  3.200000e+00  0.000000e+00            15                   1  2021-03-25T22:19:00+08:00  6.000000e+00       8.750000e-01                 12  2021-03-25T22:00:00+08:00  0.000000e+00          0.000000                1               1    8.750000e-01
2     MG01                             OSLinux-CPU_CPU_CPUWio          30  4.211209e-01  2.264500e-02            30                   2  2021-03-25T22:10:00+08:00  7.691000e-01       8.263162e-01                  2  2021-03-25T22:22:00+08:00  1.660000e-02          0.266946                1               1    8.263162e-01
3     MG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps          14  2.700000e+00  0.000000e+00            14                   1  2021-03-25T22:03:00+08:00  4.000000e+00       4.814815e-01                 11  2021-03-25T22:01:00+08:00  0.000000e+00          0.000000                1               1    4.814815e-01
4     MG01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite          15  7.077200e+02  2.697867e+02            15                   1  2021-03-25T22:02:00+08:00  8.536000e+02       2.061267e-01                  1  2021-03-25T22:19:00+08:00  2.640000e+02          0.021449                1               1    2.061267e-01
5     MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...          15  8.600000e-01  0.000000e+00            15                   1  2021-03-25T22:25:00+08:00  1.000000e+00       1.627907e-01                  9  2021-03-25T22:00:00+08:00  0.000000e+00          0.000000                1               1    1.627907e-01
6     MG01                            OSLinux-CPU_CPU_CPULoad          30  1.257500e+00  1.000000e+00            30                   2  2021-03-25T22:05:00+08:00  1.410000e+00       1.212724e-01                  5  2021-03-25T22:03:00+08:00  1.000000e+00          0.000000                1               2    1.212724e-01
7     MG01                    OSLinux-CPU_CPU-0_SingleCpuUtil          11  3.212150e+00  1.472200e+00            11                   1  2021-03-25T22:04:00+08:00  3.578600e+00       1.140825e-01                  1  2021-03-25T22:18:00+08:00  1.313800e+00          0.107594                1               1    1.140825e-01
8     MG01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite          15  1.370667e+01  7.080000e+00            15                   1  2021-03-25T22:04:00+08:00  1.520000e+01       1.089494e-01                  1  2021-03-25T22:08:00+08:00  6.800000e+00          0.039548                1               1    1.089494e-01
9     MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRea...           9  7.698000e+02  2.736000e+02             9                   1  2021-03-25T22:03:00+08:00  8.530000e+02       1.080800e-01                  1  2021-03-25T22:21:00+08:00  2.720000e+02          0.005848                1               1    1.080800e-01
10    MG01                         OSLinux-CPU_CPU_CPUSysTime          30  7.429855e-01  3.989200e-01            30                   2  2021-03-25T22:01:00+08:00  8.147000e-01       9.652200e-02                  2  2021-03-25T22:12:00+08:00  3.700000e-01          0.072496                1               1    9.652200e-02
11    MG01    OSLinux-OSLinux_NETWORK_ens160_NETBandwidthUtil           9  4.340000e-03  1.140000e-03             9                   1  2021-03-25T22:05:00+08:00  4.700000e-03       8.294931e-02                  1  2021-03-25T22:17:00+08:00  1.100000e-03          0.035088                1               1    8.294931e-02
12    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...          10  1.665000e+01  6.450000e+00            10                   1  2021-03-25T22:21:00+08:00  1.800000e+01       8.108108e-02                  1  2021-03-25T22:09:00+08:00  6.000000e+00          0.069767                1               1    8.108108e-02
13    MG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTps          15  1.035000e+00  5.283332e-01            15                   1  2021-03-25T22:04:00+08:00  1.116667e+00       7.890521e-02                  1  2021-03-25T22:08:00+08:00  5.166670e-01          0.022081                1               1    7.890521e-02
14    MG01          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed          31  9.629094e+08  2.559621e+08            31                   2  2021-03-25T22:03:00+08:00  1.035440e+09       7.532439e-02                  2  2021-03-25T22:04:00+08:00  2.454618e+08          0.041023                1               1    7.532439e-02
15    MG01    OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec          14  3.540135e+05  1.269603e+05            14                   1  2021-03-25T22:01:00+08:00  3.793960e+05       7.169924e-02                  1  2021-03-25T22:03:00+08:00  1.247360e+05          0.017520                1               1    7.169924e-02
16    MG01          JVM-Operating System_7779_JVM_JVM_CPULoad          17  1.131920e+00  5.965600e-01            17                   1  2021-03-25T22:02:00+08:00  1.206000e+00       6.544632e-02                  1  2021-03-25T22:21:00+08:00  5.864000e-01          0.017031                1               1    6.544632e-02
17    MG01                    OSLinux-CPU_CPU-2_SingleCpuUtil          10  1.000000e+02  1.604860e+00            10                   3  2021-03-25T22:02:00+08:00  1.000000e+02       0.000000e+00                  1  2021-03-25T22:22:00+08:00  1.529800e+00          0.046770                1               1    4.677043e-02
18    MG01    OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum          19  2.510500e+03  1.022300e+03            19                   1  2021-03-25T22:02:00+08:00  2.596000e+03       3.405696e-02                  1  2021-03-25T22:20:00+08:00  9.800000e+02          0.041377                1               1    4.137729e-02
19    MG01                    OSLinux-CPU_CPU-1_SingleCpuUtil          10  2.723035e+00  1.410950e+00            10                   1  2021-03-25T22:12:00+08:00  2.746300e+00       8.543776e-03                  1  2021-03-25T22:24:00+08:00  1.380800e+00          0.021369                1               1    2.136858e-02,   cmdb_id                                        kpi_name  high_streak_max  low_streak_max  series_max_sev
0    MG01  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT               29              14    1.000000e+09
1    MG01                         OSLinux-CPU_CPU_CPULoad                1               2    1.212724e-01
2    MG01              OSLinux-OSLinux_ZABBIX_Host_Uptime                1               2    1.877658e-05, {'MG01__OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT':         timestamp              timestamp_dt cmdb_id                                        kpi_name  value  high_flag  low_flag
61     1616680800  2021-03-25 22:00:00+0800    MG01  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    0.0       True      True
1182   1616680860  2021-03-25 22:01:00+0800    MG01  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    0.0       True      True
2097   1616680920  2021-03-25 22:02:00+0800    MG01  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    0.0       True      True
3218   1616680980  2021-03-25 22:03:00+0800    MG01  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    0.0       True      True
3995   1616681040  2021-03-25 22:04:00+0800    MG01  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    0.0       True      True
5088   1616681100  2021-03-25 22:05:00+0800    MG01  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    0.0       True      True
5941   1616681160  2021-03-25 22:06:00+0800    MG01  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    0.0       True      True
6861   1616681220  2021-03-25 22:07:00+0800    MG01  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    0.0       True      True
7786   1616681280  2021-03-25 22:08:00+0800    MG01  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    0.0       True      True
8904   1616681340  2021-03-25 22:09:00+0800    MG01  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    0.0       True      True
9680   1616681400  2021-03-25 22:10:00+0800    MG01  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    0.0       True      True
10668  1616681460  2021-03-25 22:11:00+0800    MG01  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    0.0       True      True
11717  1616681520  2021-03-25 22:12:00+0800    MG01  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    0.0       True      True
12793  1616681580  2021-03-25 22:13:00+0800    MG01  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    0.0       True      True
13629  1616681640  2021-03-25 22:14:00+0800    MG01  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    1.0       True     False
14605  1616681700  2021-03-25 22:15:00+0800    MG01  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    0.0       True      True
15562  1616681760  2021-03-25 22:16:00+0800    MG01  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    0.0       True      True
16542  1616681820  2021-03-25 22:17:00+0800    MG01  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    0.0       True      True
17443  1616681880  2021-03-25 22:18:00+0800    MG01  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    0.0       True      True
18464  1616681940  2021-03-25 22:19:00+0800    MG01  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    0.0       True      True, 'MG01__OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps':         timestamp              timestamp_dt cmdb_id                                        kpi_name  value  high_flag  low_flag
48     1616680800  2021-03-25 22:00:00+0800    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    0.0      False      True
3981   1616681040  2021-03-25 22:04:00+0800    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    0.0      False      True
7770   1616681280  2021-03-25 22:08:00+0800    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    0.0      False      True
9667   1616681400  2021-03-25 22:10:00+0800    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    0.0      False      True
11702  1616681520  2021-03-25 22:12:00+0800    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    0.0      False      True
13617  1616681640  2021-03-25 22:14:00+0800    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    0.0      False      True
15555  1616681760  2021-03-25 22:16:00+0800    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    0.0      False      True
18447  1616681940  2021-03-25 22:19:00+0800    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    6.0       True     False
20363  1616682060  2021-03-25 22:21:00+0800    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    0.0      False      True
22294  1616682180  2021-03-25 22:23:00+0800    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    0.0      False      True
24282  1616682300  2021-03-25 22:25:00+0800    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    0.0      False      True
26150  1616682420  2021-03-25 22:27:00+0800    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    0.0      False      True
28044  1616682540  2021-03-25 22:29:00+0800    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    0.0      False      True, 'MG01__OSLinux-CPU_CPU_CPUWio':         timestamp              timestamp_dt cmdb_id                kpi_name     value  high_flag  low_flag
9664   1616681400  2021-03-25 22:10:00+0800    MG01  OSLinux-CPU_CPU_CPUWio  0.507129       True     False
21302  1616682120  2021-03-25 22:22:00+0800    MG01  OSLinux-CPU_CPU_CPUWio  0.016600      False      True
23275  1616682240  2021-03-25 22:24:00+0800    MG01  OSLinux-CPU_CPU_CPUWio  0.020800      False      True
26111  1616682420  2021-03-25 22:27:00+0800    MG01  OSLinux-CPU_CPU_CPUWio  0.769100       True     False, 'MG01__OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps':         timestamp              timestamp_dt cmdb_id                                        kpi_name  value  high_flag  low_flag
1176   1616680860  2021-03-25 22:01:00+0800    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps    0.0      False      True
3214   1616680980  2021-03-25 22:03:00+0800    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps    4.0       True     False
5081   1616681100  2021-03-25 22:05:00+0800    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps    0.0      False      True
8897   1616681340  2021-03-25 22:09:00+0800    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps    0.0      False      True
10665  1616681460  2021-03-25 22:11:00+0800    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps    0.0      False      True
14600  1616681700  2021-03-25 22:15:00+0800    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps    0.0      False      True
16530  1616681820  2021-03-25 22:17:00+0800    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps    0.0      False      True
18454  1616681940  2021-03-25 22:19:00+0800    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps    0.0      False      True
20371  1616682060  2021-03-25 22:21:00+0800    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps    0.0      False      True
22302  1616682180  2021-03-25 22:23:00+0800    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps    0.0      False      True
24290  1616682300  2021-03-25 22:25:00+0800    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps    0.0      False      True
26158  1616682420  2021-03-25 22:27:00+0800    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps    0.0      False      True, 'MG01__OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite':         timestamp              timestamp_dt cmdb_id                                          kpi_name  value  high_flag  low_flag
2092   1616680920  2021-03-25 22:02:00+0800    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite  853.6       True     False
18459  1616681940  2021-03-25 22:19:00+0800    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite  264.0      False      True})```
```

In [ ]:
# In[10]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse kernel variables created earlier:
# window_merged (metric window rows with P95/P05/high_flag/low_flag)
# trace_window_flagged (trace window rows with P95/P05/high_flag/low_flag)

# Defensive access
wm = window_merged.copy()            # metric window rows with flags
twf = trace_window_flagged.copy()    # trace window rows with flags

# Ensure timestamp ints
wm['timestamp'] = wm['timestamp'].astype(int)
twf['timestamp'] = twf['timestamp'].astype(int)

# Convert timestamps to timestamp_dt (UTC+8) for any extracted results
def attach_dt(df):
    df = df.copy()
    df['timestamp_dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S%z')
    return df

# 1) MG01 network metrics selection
mg01_net_mask = (wm['cmdb_id'] == 'MG01')
# explicit targets
explicit_kpis = [
    'OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec',
    'OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT',
    'OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum'
]
# find kpis containing 'NETWORK' or 'NET' (case-sensitive as in data)
cand_kpis = wm[mg01_net_mask]['kpi_name'].dropna().unique()
network_matches = [k for k in cand_kpis if ('NETWORK' in k) or ('NET' in k)]
# ensure explicit are included and deduplicate, preserve order: explicit first then others
selected_kpis = []
for k in explicit_kpis + network_matches:
    if k not in selected_kpis and k in cand_kpis:
        selected_kpis.append(k)
# limit to top 10 matches
selected_kpis = selected_kpis[:10]

# For each selected KPI, extract flagged high rows, compute counts and earliest high ts, store full dfs
mg_net_summaries = []
mg_net_flagged_list = []

mg01_network_flagged_full_parts = {}

for kpi in selected_kpis:
    mask = (wm['cmdb_id'] == 'MG01') & (wm['kpi_name'] == kpi) & (wm['high_flag'] == True)
    flagged = wm.loc[mask].sort_values('timestamp', ascending=True).copy()
    flagged = attach_dt(flagged)
    high_count = int(flagged.shape[0])
    earliest_high_ts = None
    if high_count > 0:
        earliest_high_ts = pd.to_datetime(int(flagged['timestamp'].min()), unit='s', utc=True).tz_convert(tz).isoformat()
    mg_net_summaries.append({
        'cmdb_id': 'MG01',
        'kpi_name': kpi,
        'high_flagged_count': high_count,
        'earliest_high_ts': earliest_high_ts
    })
    # store full per-series df
    key = f"MG01__{kpi}"
    mg01_network_flagged_full_parts[key] = flagged
    # keep a preview (up to 20 rows) for display
    mg_net_flagged_list.append(flagged.head(20))

# concat previews into one small DataFrame for display (include series key)
if mg_net_flagged_list:
    mg_net_preview = pd.concat(
        [df.assign(series_key=k) for k, df in zip(mg01_network_flagged_full_parts.keys(), mg_net_flagged_list)],
        ignore_index=True
    ).loc[:, ['series_key','timestamp','timestamp_dt','cmdb_id','kpi_name','value','high_flag','low_flag']]
else:
    mg_net_preview = pd.DataFrame(columns=['series_key','timestamp','timestamp_dt','cmdb_id','kpi_name','value','high_flag','low_flag'])

mg_net_summary = pd.DataFrame(mg_net_summaries)

# full concatenated flagged rows for MG01 network KPIs (stored for follow-up)
if mg01_network_flagged_full_parts:
    mg01_network_flagged_full = pd.concat(mg01_network_flagged_full_parts.values(), ignore_index=True)
else:
    mg01_network_flagged_full = pd.DataFrame(columns=wm.columns.tolist() + ['timestamp_dt'])

# 2) Traces related to MG01 (specified list)
trace_targets = [
    ('Tomcat01','trace.to_MG01.duration_mean'),
    ('MG01','trace.from_Tomcat01.duration_mean'),
    ('dockerA2','trace.from_MG01.duration_mean'),
    ('dockerB2','trace.from_MG01.duration_mean')
]

trace_summaries = []
trace_flagged_parts = {}

trace_preview_list = []

for cm, tname in trace_targets:
    mask = (twf['cmdb_id'] == cm) & (twf['trace_name'] == tname) & (twf['high_flag'] == True)
    flagged_t = twf.loc[mask].sort_values('timestamp', ascending=True).copy()
    flagged_t = attach_dt(flagged_t)
    high_count = int(flagged_t.shape[0])
    earliest_high_ts = None
    if high_count > 0:
        earliest_high_ts = pd.to_datetime(int(flagged_t['timestamp'].min()), unit='s', utc=True).tz_convert(tz).isoformat()
    trace_summaries.append({
        'cmdb_id': cm,
        'trace_name': tname,
        'high_flagged_count': high_count,
        'earliest_high_ts': earliest_high_ts
    })
    key = f"{cm}__{tname}"
    trace_flagged_parts[key] = flagged_t
    trace_preview_list.append(flagged_t.head(20))

# Concise preview table for traces
if trace_preview_list:
    trace_preview = pd.concat([df.assign(series_key=k) for k, df in zip(trace_flagged_parts.keys(), trace_preview_list)], ignore_index=True)
    trace_preview = trace_preview.loc[:, ['series_key','timestamp','timestamp_dt','cmdb_id','trace_name','value','high_flag','low_flag']]
else:
    trace_preview = pd.DataFrame(columns=['series_key','timestamp','timestamp_dt','cmdb_id','trace_name','value','high_flag','low_flag'])

trace_summary = pd.DataFrame(trace_summaries)

# full concatenated flagged trace rows for stored targets
if trace_flagged_parts:
    mg01_trace_flagged_full = pd.concat(trace_flagged_parts.values(), ignore_index=True)
else:
    mg01_trace_flagged_full = pd.DataFrame(columns=twf.columns.tolist() + ['timestamp_dt'])

# Final compact outputs (for display): summaries and previews, plus shapes of full stored subsets
mg_net_summary, mg_net_preview, mg01_network_flagged_full.shape, trace_summary, trace_preview, mg01_trace_flagged_full.shape

```
Out[10]:
```
Summary (plain English):

- What I extracted and stored:
  - All flagged MG01 network metric rows (high_flag or low_flag) for selected network KPIs — stored in kernel variable mg01_network_flagged_full (shape reported: 94 rows × 10 cols) and summarized in mg_net_summary / mg_net_preview.
  - Flagged trace rows for the requested traces — stored in kernel variable mg01_trace_flagged_full and summarized in trace_summary / trace_preview.

- MG01 network metric highlights (selected KPIs, flagged-high counts and earliest high timestamps):
  1. OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec — high_count 1, earliest high 2021-03-25T22:01:00+08 (example flagged value 379396.0)
  2. OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT — high_count 30, earliest high 2021-03-25T22:00:00+08 (sustained flagged run across the window)
  3. OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum — high_count 1, earliest high 2021-03-25T22:02:00+08
  4. OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT — high_count 30, earliest high 2021-03-25T22:00:00+08
  5. OSLinux-OSLinux_NETWORK_ens160_NETBandwidthUtil — high_count 1, earliest high 2021-03-25T22:05:00+08
  6. OSLinux-OSLinux_NETWORK_ens160_NETInErr — high_count 9, earliest high 2021-03-25T22:01:00+08
  7. OSLinux-OSLinux_NETWORK_ens160_NETOutErr — high_count 9, earliest high 2021-03-25T22:01:00+08
  8. OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn — high_count 1, earliest high 2021-03-25T22:25:00+08
  9. OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut — high_count 1, earliest high 2021-03-25T22:25:00+08
 10. OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc — high_count 11, earliest high 2021-03-25T22:02:00+08
  - (Selection limited to top ~10 network-related KPIs; full flagged rows stored)

- MG01 trace highlights (requested traces):
  - Tomcat01 — trace.to_MG01.duration_mean: high_count 2, earliest high 2021-03-25T22:01:00+08 (flagged values e.g., 0.988390 and 0.135047)
  - MG01 — trace.from_Tomcat01.duration_mean: high_count 2, earliest high 2021-03-25T22:01:00+08 (same two timestamps/values on MG01 side)
  - dockerA2 — trace.from_MG01.duration_mean: high_count 2, earliest high 2021-03-25T22:01:00+08 (example flagged value 0.891886)
  - dockerB2 — trace.from_MG01.duration_mean: high_count 2, earliest high 2021-03-25T22:01:00+08 (example flagged value 1.925052)
  - Up to 20 flagged trace rows per series were extracted and a preview stored.

- Notable patterns / interpretation:
  - TCP-CLOSE-WAIT and TCP-FIN-WAIT on MG01 show sustained flagged runs across the window (many flagged samples starting at 22:00+08) — suggests a sustained network/connection-state condition.
  - Packet error metrics (NETInErr / NETOutErr / NETInErrPrc) show multiple flagged points starting ~22:01+08 — indicates packet errors around the same early window time.
  - A large NETKBTotalPerSec spike at 22:01 on MG01 was flagged (example value ~379,396) — suggests a burst of traffic.
  - Trace-side high points for Tomcat01 <-> MG01 and from MG01 to docker nodes cluster at ~22:01+08, aligning with the early-network anomalies.
  - dockerB2 trace had a notably large flagged value (≈1.92505) at 22:01 — another strong trace anomaly to inspect.

- Stored kernel variables for follow-up:
  - mg01_network_flagged_full — full flagged rows for MG01 selected network KPIs (for detailed inspection).
  - mg_net_summary, mg_net_preview — compact summaries and previews.
  - mg01_trace_flagged_full — full flagged rows for the requested traces.
  - trace_summary, trace_preview — compact trace summaries and previews.

- Suggested next steps:
  1. Inspect full flagged rows (mg01_network_flagged_full) around 22:00–22:05+08 for TCP CLOSE/FIN states and packet errors.
  2. Correlate the early ~22:01+08 trace spikes (Tomcat01↔MG01 and MG01->docker nodes) with the network spikes and errors on MG01.
  3. Check application logs and upstream/downstream components for connection errors or traffic bursts at ~22:00–22:05+08.

If you want, I can now:
- display the full flagged rows for any one specific MG01 KPI or trace series, or
- produce a timeline correlation (merged table) for MG01 network metrics and the Tomcat01↔MG01 trace samples around 22:00–22:05+08.

The original code execution output of IPython Kernel is also provided below for reference:

(  cmdb_id                                         kpi_name  high_flagged_count           earliest_high_ts
0    MG01  OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec                   1  2021-03-25T22:01:00+08:00
1    MG01   OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT                  30  2021-03-25T22:00:00+08:00
2    MG01  OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum                   1  2021-03-25T22:02:00+08:00
3    MG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT                  30  2021-03-25T22:00:00+08:00
4    MG01  OSLinux-OSLinux_NETWORK_ens160_NETBandwidthUtil                   1  2021-03-25T22:05:00+08:00
5    MG01          OSLinux-OSLinux_NETWORK_ens160_NETInErr                   9  2021-03-25T22:01:00+08:00
6    MG01         OSLinux-OSLinux_NETWORK_ens160_NETOutErr                   9  2021-03-25T22:01:00+08:00
7    MG01      OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn                   1  2021-03-25T22:25:00+08:00
8    MG01     OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut                   1  2021-03-25T22:25:00+08:00
9    MG01       OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc                  11  2021-03-25T22:02:00+08:00,                                            series_key   timestamp              timestamp_dt cmdb_id                                         kpi_name     value  high_flag  low_flag
0   MG01__OSLinux-OSLinux_NETWORK_ens160_NETKBTota...  1616680860  2021-03-25 22:01:00+0800    MG01  OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec  379396.0       True     False
1   MG01__OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOS...  1616680800  2021-03-25 22:00:00+0800    MG01   OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT       0.0       True      True
2   MG01__OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOS...  1616680860  2021-03-25 22:01:00+0800    MG01   OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT       0.0       True      True
3   MG01__OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOS...  1616680920  2021-03-25 22:02:00+0800    MG01   OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT       0.0       True      True
4   MG01__OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOS...  1616680980  2021-03-25 22:03:00+0800    MG01   OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT       0.0       True      True
..                                                ...         ...                       ...     ...                                              ...       ...        ...       ...
69   MG01__OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc  1616682000  2021-03-25 22:20:00+0800    MG01       OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc       0.0       True      True
70   MG01__OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc  1616682120  2021-03-25 22:22:00+0800    MG01       OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc       0.0       True      True
71   MG01__OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc  1616682240  2021-03-25 22:24:00+0800    MG01       OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc       0.0       True      True
72   MG01__OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc  1616682360  2021-03-25 22:26:00+0800    MG01       OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc       0.0       True      True
73   MG01__OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc  1616682600  2021-03-25 22:30:00+0800    MG01       OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc       0.0       True      True

[74 rows x 8 columns], (94, 10),     cmdb_id                         trace_name  high_flagged_count           earliest_high_ts
0  Tomcat01        trace.to_MG01.duration_mean                   2  2021-03-25T22:01:00+08:00
1      MG01  trace.from_Tomcat01.duration_mean                   2  2021-03-25T22:01:00+08:00
2  dockerA2      trace.from_MG01.duration_mean                   2  2021-03-25T22:01:00+08:00
3  dockerB2      trace.from_MG01.duration_mean                   2  2021-03-25T22:01:00+08:00,                                 series_key   timestamp              timestamp_dt   cmdb_id                         trace_name     value  high_flag  low_flag
0    Tomcat01__trace.to_MG01.duration_mean  1616680860  2021-03-25 22:01:00+0800  Tomcat01        trace.to_MG01.duration_mean  0.988390       True     False
1    Tomcat01__trace.to_MG01.duration_mean  1616681580  2021-03-25 22:13:00+0800  Tomcat01        trace.to_MG01.duration_mean  0.135047       True     False
2  MG01__trace.from_Tomcat01.duration_mean  1616680860  2021-03-25 22:01:00+0800      MG01  trace.from_Tomcat01.duration_mean  0.988390       True     False
3  MG01__trace.from_Tomcat01.duration_mean  1616681580  2021-03-25 22:13:00+0800      MG01  trace.from_Tomcat01.duration_mean  0.135047       True     False
4  dockerA2__trace.from_MG01.duration_mean  1616680860  2021-03-25 22:01:00+0800  dockerA2      trace.from_MG01.duration_mean  0.891886       True     False
5  dockerA2__trace.from_MG01.duration_mean  1616682060  2021-03-25 22:21:00+0800  dockerA2      trace.from_MG01.duration_mean  0.074194       True     False
6  dockerB2__trace.from_MG01.duration_mean  1616680860  2021-03-25 22:01:00+0800  dockerB2      trace.from_MG01.duration_mean  1.925052       True     False
7  dockerB2__trace.from_MG01.duration_mean  1616681220  2021-03-25 22:07:00+0800  dockerB2      trace.from_MG01.duration_mean  0.306703       True     False, (8, 10))

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```